# 08 — Modelagem e Validação de Machine Learning

## 1. Objetivo da etapa

Este notebook tem como objetivo desenvolver, treinar, validar e comparar modelos supervisionados de Machine Learning para prever se um aluno será considerado alfabetizado ou não alfabetizado.

A modelagem será realizada a partir dos conjuntos de treino, validação e teste definidos e persistidos na etapa anterior, preservando a separação territorial por município e as decisões adotadas para prevenção de data leakage.

O pré-processamento será integrado diretamente às pipelines dos modelos por meio da arquitetura reutilizável definida em `src/preprocessing.py`, contemplando:

- engenharia de atributos por meio de indicadores de ausência;
- imputação de valores ausentes nas variáveis numéricas;
- transformação das variáveis categóricas;
- padronização das variáveis numéricas quando requerida pela família do modelo;
- integração entre pré-processamento e estimador.

O conjunto de treino será utilizado para o aprendizado dos parâmetros dos modelos e das transformações associadas.

O conjunto de validação será utilizado para comparação de alternativas, análise das métricas, seleção de modelos e demais decisões da etapa de desenvolvimento.

O conjunto de teste permanecerá isolado durante esse processo e será utilizado somente após a definição da estratégia final, permitindo uma avaliação independente da capacidade de generalização do modelo selecionado.

## 2. Carregamento dos conjuntos de modelagem

Os conjuntos de treino, validação e teste foram definidos e persistidos na etapa anterior do projeto.

Nesta etapa, esses artefatos serão carregados diretamente do diretório `tech_challenge_fase3/modelagem`, preservando as partições territoriais já estabelecidas e evitando a repetição do processo de separação dos dados.

Cada arquivo preditivo contém as 16 features selecionadas para modelagem e o target `in_alfabetizado`.

Também serão carregados os arquivos auxiliares de rastreabilidade gerados no Notebook 07. Esses arquivos preservam `id_aluno`, `id_escola` e `co_municipio` e permanecem completamente separados das features utilizadas pelos modelos.

Após o carregamento, serão verificadas as dimensões e a correspondência estrutural entre os conjuntos preditivos e seus respectivos arquivos auxiliares.


In [0]:
# Objetivo:
#
# Carregar os conjuntos de treino, validação
# e teste persistidos na etapa anterior.
#
# Justificativa:
#
# A utilização dos artefatos já definidos e
# validados garante que a modelagem utilize
# exatamente as mesmas partições territoriais
# estabelecidas no Notebook 07.
#
# Dessa forma, não é necessário repetir o
# processo de separação dos dados.
#
# Ação:
#
# Carrega os três arquivos Parquet produzidos
# na Fase 3 e verifica suas dimensões.

import pandas as pd

FASE3_ROOT = (
    "/Volumes/workspace/default/vol_trio_drive/"
    "projetos/fiap/tech_challenge_fase3"
)

MODELAGEM_PATH = f"{FASE3_ROOT}/modelagem"

df_treino = pd.read_parquet(
    f"{MODELAGEM_PATH}/treino.parquet"
)

df_validacao = pd.read_parquet(
    f"{MODELAGEM_PATH}/validacao.parquet"
)

df_teste = pd.read_parquet(
    f"{MODELAGEM_PATH}/teste.parquet"
)


pd.Series({
    "registros_treino": df_treino.shape[0],
    "colunas_treino": df_treino.shape[1],
    "registros_validacao": df_validacao.shape[0],
    "colunas_validacao": df_validacao.shape[1],
    "registros_teste": df_teste.shape[0],
    "colunas_teste": df_teste.shape[1]
})

### 2.1 Carregamento das variáveis auxiliares de rastreabilidade

Além dos conjuntos preditivos, serão carregados os arquivos auxiliares gerados no Notebook 07.

Esses arquivos preservam `id_aluno`, `id_escola` e `co_municipio`, permitindo associar posteriormente as previsões dos modelos às observações originais.

As variáveis auxiliares não serão utilizadas como features e permanecerão fora das pipelines de Machine Learning.

Sua finalidade é exclusivamente garantir rastreabilidade e apoiar análises posteriores por aluno, escola e município.


In [0]:
# Objetivo:
#
# Carregar as variáveis auxiliares de
# rastreabilidade geradas no Notebook 07.
#
# Justificativa:
#
# Os identificadores de aluno, escola e município
# não participam do treinamento dos modelos, mas
# precisam permanecer associados às observações
# para permitir a rastreabilidade das previsões.
#
# Ação:
#
# Carrega os três arquivos auxiliares correspondentes
# aos conjuntos de treino, validação e teste.

auxiliares_treino = pd.read_parquet(
    f"{MODELAGEM_PATH}/auxiliares_treino.parquet"
)

auxiliares_validacao = pd.read_parquet(
    f"{MODELAGEM_PATH}/auxiliares_validacao.parquet"
)

auxiliares_teste = pd.read_parquet(
    f"{MODELAGEM_PATH}/auxiliares_teste.parquet"
)


pd.Series({
    "auxiliares_treino": auxiliares_treino.shape,
    "auxiliares_validacao": auxiliares_validacao.shape,
    "auxiliares_teste": auxiliares_teste.shape
})


### 2.2 Validação da correspondência entre dados preditivos e auxiliares

Os arquivos auxiliares foram persistidos no Notebook 07 utilizando exatamente a mesma ordem de linhas dos respectivos conjuntos preditivos.

Nesta etapa será validado o contrato estrutural dessa persistência: quantidade de registros, índices após a leitura, presença das três colunas auxiliares esperadas e ausência desses identificadores nos conjuntos utilizados pelos modelos.

A correspondência semântica entre as linhas foi estabelecida no Notebook 07 antes da persistência; aqui é verificado que essa estrutura permaneceu íntegra após a leitura dos Parquets.


In [0]:
# Objetivo:
#
# Validar a correspondência estrutural entre os
# conjuntos preditivos e os arquivos auxiliares.
#
# Justificativa:
#
# As previsões só podem ser associadas aos alunos
# se os arquivos auxiliares preservarem exatamente
# a mesma quantidade e ordem de registros definida
# no Notebook 07.
#
# Também é necessário confirmar que os identificadores
# permanecem fora dos dados utilizados pelos modelos.
#
# Ação:
#
# Verifica dimensões, índices, colunas esperadas e
# ausência das variáveis auxiliares nos DataFrames
# preditivos.

colunas_auxiliares_esperadas = {
    "id_aluno",
    "id_escola",
    "co_municipio"
}

validacao_rastreabilidade = pd.Series({
    "treino_mesmo_numero_registros": (
        len(df_treino) == len(auxiliares_treino)
    ),
    "validacao_mesmo_numero_registros": (
        len(df_validacao) == len(auxiliares_validacao)
    ),
    "teste_mesmo_numero_registros": (
        len(df_teste) == len(auxiliares_teste)
    ),
    "treino_indices_alinhados": (
        df_treino.index.equals(auxiliares_treino.index)
    ),
    "validacao_indices_alinhados": (
        df_validacao.index.equals(auxiliares_validacao.index)
    ),
    "teste_indices_alinhados": (
        df_teste.index.equals(auxiliares_teste.index)
    ),
    "colunas_auxiliares_treino_ok": (
        set(auxiliares_treino.columns)
        == colunas_auxiliares_esperadas
    ),
    "colunas_auxiliares_validacao_ok": (
        set(auxiliares_validacao.columns)
        == colunas_auxiliares_esperadas
    ),
    "colunas_auxiliares_teste_ok": (
        set(auxiliares_teste.columns)
        == colunas_auxiliares_esperadas
    ),
    "auxiliares_fora_dos_dados_preditivos": (
        colunas_auxiliares_esperadas.isdisjoint(df_treino.columns)
        and colunas_auxiliares_esperadas.isdisjoint(df_validacao.columns)
        and colunas_auxiliares_esperadas.isdisjoint(df_teste.columns)
    )
})

validacao_rastreabilidade


### 2.3 Separação entre features e target

Após o carregamento dos conjuntos persistidos, será realizada a separação entre as variáveis preditoras e a variável-alvo.

As 16 features selecionadas na etapa anterior permanecerão em `X`, enquanto `in_alfabetizado` será utilizado como target (`y`).

Essa separação será realizada de forma idêntica nos conjuntos de treino, validação e teste, preservando as mesmas partições definidas anteriormente. As variáveis auxiliares permanecerão em estruturas separadas e não serão incluídas em `X`.

In [0]:
# Objetivo:
#
# Separar as features e o target nos conjuntos
# de treino, validação e teste.
#
# Justificativa:
#
# A modelagem supervisionada exige que as
# variáveis preditoras sejam separadas da
# variável-alvo utilizada para treinamento
# e avaliação dos modelos.
#
# A separação será feita de forma consistente
# nos três conjuntos já definidos.
#
# Ação:
#
# Remove o target dos DataFrames para formar
# X e extrai in_alfabetizado para formar y.

X_treino = df_treino.drop(
    columns="in_alfabetizado"
)

y_treino = df_treino[
    "in_alfabetizado"
].copy()


X_validacao = df_validacao.drop(
    columns="in_alfabetizado"
)

y_validacao = df_validacao[
    "in_alfabetizado"
].copy()


X_teste = df_teste.drop(
    columns="in_alfabetizado"
)

y_teste = df_teste[
    "in_alfabetizado"
].copy()


pd.Series({
    "X_treino": X_treino.shape,
    "y_treino": y_treino.shape,
    "X_validacao": X_validacao.shape,
    "y_validacao": y_validacao.shape,
    "X_teste": X_teste.shape,
    "y_teste": y_teste.shape
})

## 3. Integração do pré-processamento reutilizável

A arquitetura de pré-processamento definida e validada na etapa anterior foi externalizada para o módulo `src/preprocessing.py`.

Essa separação permite reutilizar a mesma lógica de engenharia de atributos e transformação dos dados durante a modelagem, evitando duplicação de código entre notebooks e mantendo o pré-processamento integrado ao fluxo de Machine Learning.

O módulo disponibiliza a função `criar_pre_processamento()`, capaz de construir a arquitetura com ou sem padronização das variáveis numéricas, de acordo com as necessidades da família de modelos utilizada.

### 3.1 Disponibilização do módulo Python no Databricks

Para que a importação de `src.preprocessing` seja independente da localização do notebook no Workspace, o arquivo reutilizável deve ser disponibilizado na estrutura oficial do projeto.

Neste projeto, os arquivos devem ser organizados da seguinte forma:

```text
/Volumes/workspace/default/vol_trio_drive/
└── projetos/fiap/tech_challenge_fase3/
    └── src/
        ├── __init__.py
        └── preprocessing.py
```

O arquivo `__init__.py` pode permanecer vazio. Sua presença identifica `src` como pacote Python. O arquivo `preprocessing.py` deve conter a versão que disponibiliza o parâmetro opcional `saida_densa`, necessário para o `HistGradientBoostingClassifier`.

A célula seguinte adiciona a raiz do projeto ao caminho de importação do Python, confirma a existência dos dois arquivos e invalida o cache de importações antes da utilização do módulo. Esse procedimento deve ser executado antes de `from src.preprocessing import criar_pre_processamento`.

In [0]:
# Objetivo:
#
# Disponibilizar o pacote src para importação
# no ambiente de execução do Databricks.
#
# Justificativa:
#
# A localização do notebook no Workspace não
# garante que a raiz do projeto esteja presente
# no sys.path do Python. A inclusão explícita
# evita erros como "No module named 'src'".
#
# Ação:
#
# Define a raiz oficial do projeto, valida os
# arquivos do pacote e adiciona essa raiz ao
# caminho de importação.

import importlib
import os
import sys


PROJECT_ROOT = FASE3_ROOT
SRC_PATH = f"{PROJECT_ROOT}/src"
INIT_FILE = f"{SRC_PATH}/__init__.py"
PREPROCESSING_FILE = (
    f"{SRC_PATH}/preprocessing.py"
)


arquivos_modulo = {
    "__init__.py": os.path.isfile(INIT_FILE),
    "preprocessing.py": os.path.isfile(
        PREPROCESSING_FILE
    )
}


if not all(arquivos_modulo.values()):
    arquivos_ausentes = [
        nome
        for nome, existe in arquivos_modulo.items()
        if not existe
    ]

    raise FileNotFoundError(
        "Arquivos ausentes em "
        f"{SRC_PATH}: {arquivos_ausentes}. "
        "Faça o upload de preprocessing.py e "
        "crie um __init__.py vazio antes de "
        "prosseguir."
    )


if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)


importlib.invalidate_caches()


print("Módulo de pré-processamento disponível.")
print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Arquivo validado: {PREPROCESSING_FILE}")

In [0]:
# Objetivo:
#
# Importar a arquitetura reutilizável de
# pré-processamento criada na etapa anterior.
#
# Justificativa:
#
# A centralização dessa lógica em um módulo
# Python evita duplicação de código e garante
# que os modelos utilizem a mesma estratégia
# de preparação dos dados.
#
# Ação:
#
# Importa a função responsável pela construção
# da arquitetura de pré-processamento e cria
# uma instância para verificar sua integração
# com o Notebook 08.

from src.preprocessing import criar_pre_processamento


pre_processamento = criar_pre_processamento(
    padronizar=False
)

pre_processamento

## 4. Estratégia de avaliação dos modelos

A modelagem será tratada como um problema de classificação binária supervisionada, tendo como objetivo identificar alunos com risco de não alfabetização.

Embora a codificação original do target `in_alfabetizado` seja:

- `0` — não alfabetizado;
- `1` — alfabetizado;

a classe `0` será considerada a classe positiva de interesse para fins de avaliação, pois representa justamente a condição que o projeto busca identificar.

### 4.1 Custo dos erros

Sob essa perspectiva, um falso negativo ocorre quando um aluno realmente não alfabetizado é classificado pelo modelo como alfabetizado.

Esse erro possui especial relevância para o problema, pois representa um aluno em situação de risco que não seria identificado para eventual priorização.

Por outro lado, um falso positivo ocorre quando um aluno alfabetizado é classificado como não alfabetizado. Nesse caso, o modelo produz um alerta desnecessário, podendo direcionar atenção ou recursos para um aluno que não pertence ao grupo de risco.

Dessa forma, a estratégia de avaliação buscará reduzir falsos negativos sem ignorar o custo associado à geração excessiva de falsos positivos.

### 4.2 Métricas de avaliação

O **recall da classe não alfabetizado (`0`)** terá papel central na avaliação dos modelos, pois mede a proporção dos alunos realmente não alfabetizados que foram corretamente identificados.

Entretanto, o recall não será analisado isoladamente. Um modelo que classifique indiscriminadamente grande parte dos alunos como pertencentes ao grupo de risco pode apresentar recall elevado e, ao mesmo tempo, gerar quantidade excessiva de falsos positivos.

Por esse motivo, a avaliação considerará também a precisão da classe de interesse e o F1-score, permitindo analisar o equilíbrio entre identificação dos alunos em risco e qualidade dos alertas produzidos.

A matriz de confusão será utilizada para traduzir as métricas em quantidades concretas de alunos corretamente e incorretamente classificados.

Métricas complementares de discriminação, como PR-AUC e ROC-AUC, também serão consideradas durante a comparação dos modelos. Outras medidas de desempenho e calibração poderão ser incorporadas nas etapas posteriores de validação conforme a necessidade da análise.

Sempre que uma métrica binária depender da definição explícita da classe positiva, a classe `0` será informada como referência, evitando que o comportamento padrão das funções seja interpretado incorretamente como avaliação da classe de interesse do projeto.

### 4.3 Regra para seleção

A seleção do modelo não será baseada em uma única métrica isolada.

Será priorizado um modelo capaz de identificar adequadamente os alunos não alfabetizados, com especial atenção à redução dos falsos negativos, mantendo simultaneamente um nível de precisão que evite volume excessivo de alertas desnecessários.

As decisões de seleção, ajuste de hiperparâmetros e definição do limiar de classificação serão realizadas utilizando os dados de treino e validação. O conjunto de teste permanecerá isolado até que essas decisões estejam congeladas.

## 5. Baseline mínimo

Antes do treinamento dos modelos preditivos, será estabelecido um baseline mínimo de desempenho.

O baseline servirá como referência para verificar se os modelos supervisionados são capazes de superar uma estratégia ingênua de classificação.

Antes de sua construção, será analisada a distribuição do target no conjunto de treino, permitindo compreender o balanceamento entre alunos alfabetizados e não alfabetizados.

Essa análise será realizada exclusivamente sobre `y_treino`, sem utilizar os conjuntos de validação e teste para decisões de desenvolvimento.

In [0]:
# Objetivo:
#
# Analisar a distribuição da variável-alvo
# no conjunto de treino.
#
# Justificativa:
#
# Antes da construção do baseline e da eventual
# aplicação de técnicas para desbalanceamento,
# é necessário conhecer a frequência e a proporção
# das classes presentes no target.
#
# A classe 0 representa os alunos não alfabetizados
# e constitui a classe positiva de interesse
# para o projeto.
#
# Ação:
#
# Calcula a quantidade e a proporção percentual
# das classes presentes em y_treino.

distribuicao_target = pd.DataFrame({
    "quantidade": y_treino.value_counts().sort_index(),
    "percentual": (
        y_treino
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )
})

distribuicao_target.index = [
    "0 - Não alfabetizado",
    "1 - Alfabetizado"
]

distribuicao_target

### 5.1 Construção e treinamento do baseline ingênuo

O primeiro modelo utilizado será o `DummyClassifier`, que servirá como baseline mínimo para a comparação dos modelos posteriores.

Será adotada a estratégia `most_frequent`, na qual o classificador aprende apenas qual é a classe mais frequente no conjunto de treino e passa a utilizá-la como previsão.

Esse modelo não busca aprender relações entre as features e o target. Sua função é estabelecer uma referência mínima de desempenho que os modelos supervisionados deverão superar.

Mesmo tratando-se de um classificador ingênuo, o modelo será integrado à arquitetura de pré-processamento por meio de uma `Pipeline` do Scikit-learn, mantendo o mesmo padrão que será utilizado nos modelos posteriores.

Como o `DummyClassifier` não depende da escala das variáveis, a padronização numérica permanecerá desativada nesta pipeline.

In [0]:
# Objetivo:
#
# Construir e treinar o baseline mínimo
# utilizando o DummyClassifier.
#
# Justificativa:
#
# O baseline ingênuo estabelece uma referência
# mínima de desempenho para os modelos que serão
# avaliados posteriormente.
#
# A estratégia most_frequent prevê sempre a classe
# mais frequente aprendida no conjunto de treino.
#
# Mesmo para o baseline, o pré-processamento será
# mantido integrado ao estimador por meio de uma
# Pipeline do Scikit-learn.
#
# Ação:
#
# Constrói uma nova arquitetura de pré-processamento,
# integra o DummyClassifier e ajusta a pipeline
# completa exclusivamente sobre o conjunto de treino.

from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline


pipeline_dummy = Pipeline(
    steps=[
        (
            "pre_processamento",
            criar_pre_processamento(
                padronizar=False
            )
        ),
        (
            "modelo",
            DummyClassifier(
                strategy="most_frequent"
            )
        )
    ]
)


pipeline_dummy.fit(
    X_treino,
    y_treino
)

### 5.2 Avaliação do baseline no conjunto de validação

Após o treinamento exclusivamente sobre o conjunto de treino, o baseline será avaliado no conjunto de validação.

O conjunto de validação contém municípios não utilizados no ajuste da pipeline, permitindo observar o comportamento do baseline fora dos grupos utilizados no treinamento.

Nesta primeira avaliação serão calculadas métricas de classificação sob a perspectiva da classe de interesse `0` — alunos não alfabetizados.

Serão analisados:

- recall da classe não alfabetizado;
- precisão da classe não alfabetizado;
- F1-score da classe não alfabetizado;
- acurácia;
- balanced accuracy;
- matriz de confusão.

O conjunto de teste permanecerá isolado e não participará desta etapa.

In [0]:
# Objetivo:
#
# Avaliar o baseline ingênuo no conjunto
# de validação.
#
# Justificativa:
#
# O conjunto de validação permite medir o
# desempenho do modelo em municípios que não
# participaram do treinamento.
#
# Como a classe positiva de interesse é 0,
# recall, precisão e F1 serão calculados
# explicitamente com pos_label=0.
#
# Ação:
#
# Realiza as previsões no conjunto de validação
# e calcula as principais métricas de
# classificação do baseline.

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score
)


y_pred_dummy = pipeline_dummy.predict(
    X_validacao
)


metricas_dummy = pd.Series({
    "accuracy": accuracy_score(
        y_validacao,
        y_pred_dummy
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_validacao,
        y_pred_dummy
    ),
    "recall_nao_alfabetizado": recall_score(
        y_validacao,
        y_pred_dummy,
        pos_label=0
    ),
    "precision_nao_alfabetizado": precision_score(
        y_validacao,
        y_pred_dummy,
        pos_label=0,
        zero_division=0
    ),
    "f1_nao_alfabetizado": f1_score(
        y_validacao,
        y_pred_dummy,
        pos_label=0,
        zero_division=0
    )
})


metricas_dummy

### 5.3 Matriz de confusão do baseline

A matriz de confusão será utilizada para analisar os erros e acertos do baseline em termos absolutos, permitindo traduzir as métricas de classificação em quantidade de alunos.

Como a classe positiva de interesse do projeto é `0` — não alfabetizado — a interpretação da matriz seguirá essa perspectiva.

Serão identificados:

- verdadeiro positivo (VP): aluno não alfabetizado corretamente identificado como não alfabetizado;
- falso negativo (FN): aluno não alfabetizado incorretamente classificado como alfabetizado;
- falso positivo (FP): aluno alfabetizado incorretamente classificado como não alfabetizado;
- verdadeiro negativo (VN): aluno alfabetizado corretamente identificado como alfabetizado.

Entre esses erros, o falso negativo possui especial relevância para o projeto, pois representa um aluno realmente não alfabetizado que deixou de ser identificado pelo modelo.

In [0]:
# Objetivo:
#
# Construir e interpretar a matriz de confusão
# do baseline no conjunto de validação.
#
# Justificativa:
#
# A matriz de confusão permite traduzir as
# métricas em quantidades concretas de alunos
# classificados correta e incorretamente.
#
# Como a classe positiva de interesse é 0,
# a interpretação será realizada sob a
# perspectiva dos alunos não alfabetizados.
#
# Ação:
#
# Calcula a matriz de confusão com ordem
# explícita das classes e extrai VP, FN,
# FP e VN.

from sklearn.metrics import confusion_matrix


matriz_dummy = confusion_matrix(
    y_validacao,
    y_pred_dummy,
    labels=[0, 1]
)


vp = matriz_dummy[0, 0]
fn = matriz_dummy[0, 1]
fp = matriz_dummy[1, 0]
vn = matriz_dummy[1, 1]


pd.Series({
    "VP_nao_alfabetizado": vp,
    "FN_nao_alfabetizado": fn,
    "FP_nao_alfabetizado": fp,
    "VN_nao_alfabetizado": vn
})

## 6. Baseline explicável — Regressão Logística

Após o estabelecimento do baseline ingênuo, será construída uma Regressão Logística como primeiro modelo capaz de aprender relações entre as features e o target.

A Regressão Logística constitui um baseline explicável e fornece uma referência linear para comparação com modelos mais complexos nas etapas posteriores.

Como o algoritmo é sensível à escala das variáveis e utiliza regularização, o pré-processamento será configurado com padronização das features numéricas.

Também será utilizada inicialmente a estratégia `class_weight="balanced"`, atribuindo pesos inversamente proporcionais à frequência das classes durante o treinamento.

Essa configuração busca aumentar a importância relativa da classe menos frequente — alunos não alfabetizados — sem modificar fisicamente a distribuição dos dados de treino.

O desempenho obtido será posteriormente comparado com outras configurações e modelos, considerando especialmente o recall da classe não alfabetizado e seu trade-off com precisão e falsos positivos.

In [0]:
# Objetivo:
#
# Construir e treinar o primeiro baseline
# explicável utilizando Regressão Logística.
#
# Justificativa:
#
# A Regressão Logística permite estabelecer
# uma referência linear e interpretável para
# comparação com modelos mais complexos.
#
# Como o algoritmo utiliza regularização e é
# sensível à escala das variáveis, a padronização
# será ativada no pré-processamento.
#
# O balanceamento das classes será considerado
# inicialmente para aumentar a importância
# relativa da classe menos frequente durante
# o treinamento.
#
# Ação:
#
# Constrói uma pipeline completa contendo
# pré-processamento com padronização e
# Regressão Logística balanceada e realiza
# o ajuste exclusivamente no conjunto de treino.

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


pipeline_logistica_balanceada = Pipeline(
    steps=[
        (
            "pre_processamento",
            criar_pre_processamento(
                padronizar=True
            )
        ),
        (
            "modelo",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)


pipeline_logistica_balanceada.fit(
    X_treino,
    y_treino
)

### 6.1 Avaliação da Regressão Logística balanceada

Após o treinamento, a Regressão Logística balanceada será avaliada no conjunto de validação utilizando as mesmas métricas adotadas para o baseline ingênuo.

A manutenção do mesmo protocolo de avaliação permite comparar os modelos de forma consistente.

Como a classe positiva de interesse permanece sendo `0` — não alfabetizado — recall, precisão e F1-score serão calculados explicitamente sob essa perspectiva.

Nesta etapa serão avaliados:

- acurácia;
- balanced accuracy;
- recall da classe não alfabetizado;
- precisão da classe não alfabetizado;
- F1-score da classe não alfabetizado.

O conjunto de teste permanecerá isolado.

In [0]:
# Objetivo:
#
# Avaliar a Regressão Logística balanceada
# no conjunto de validação.
#
# Justificativa:
#
# A utilização das mesmas métricas aplicadas
# ao baseline ingênuo permite comparar os
# modelos sob um protocolo consistente.
#
# Como a classe positiva de interesse é 0,
# recall, precisão e F1 serão calculados
# explicitamente com pos_label=0.
#
# Ação:
#
# Realiza as previsões no conjunto de validação
# e calcula as principais métricas de
# classificação.

y_pred_logistica_balanceada = (
    pipeline_logistica_balanceada.predict(
        X_validacao
    )
)


metricas_logistica_balanceada = pd.Series({
    "accuracy": accuracy_score(
        y_validacao,
        y_pred_logistica_balanceada
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_validacao,
        y_pred_logistica_balanceada
    ),
    "recall_nao_alfabetizado": recall_score(
        y_validacao,
        y_pred_logistica_balanceada,
        pos_label=0
    ),
    "precision_nao_alfabetizado": precision_score(
        y_validacao,
        y_pred_logistica_balanceada,
        pos_label=0,
        zero_division=0
    ),
    "f1_nao_alfabetizado": f1_score(
        y_validacao,
        y_pred_logistica_balanceada,
        pos_label=0,
        zero_division=0
    )
})


metricas_logistica_balanceada

### 6.2 Matriz de confusão da Regressão Logística balanceada

A matriz de confusão será utilizada para traduzir o desempenho da Regressão Logística balanceada em quantidades concretas de alunos.

A análise permitirá verificar quantos alunos não alfabetizados foram corretamente identificados e quantos permaneceram como falsos negativos, além de quantificar os falsos positivos gerados pelo modelo.

Essa interpretação é especialmente importante para avaliar o trade-off observado entre recall e precisão da classe de interesse.

A matriz seguirá a mesma convenção utilizada no baseline, considerando `0` — não alfabetizado — como classe positiva de interesse.

In [0]:
# Objetivo:
#
# Construir e interpretar a matriz de confusão
# da Regressão Logística balanceada.
#
# Justificativa:
#
# A matriz de confusão permite traduzir o
# desempenho do modelo em quantidades concretas
# de alunos e compreender o trade-off entre
# falsos negativos e falsos positivos.
#
# A mesma convenção utilizada no baseline será
# mantida, considerando a classe 0 como positiva
# de interesse.
#
# Ação:
#
# Calcula a matriz de confusão no conjunto de
# validação e extrai VP, FN, FP e VN.

matriz_logistica_balanceada = confusion_matrix(
    y_validacao,
    y_pred_logistica_balanceada,
    labels=[0, 1]
)


vp_logistica = matriz_logistica_balanceada[0, 0]
fn_logistica = matriz_logistica_balanceada[0, 1]
fp_logistica = matriz_logistica_balanceada[1, 0]
vn_logistica = matriz_logistica_balanceada[1, 1]


pd.Series({
    "VP_nao_alfabetizado": vp_logistica,
    "FN_nao_alfabetizado": fn_logistica,
    "FP_nao_alfabetizado": fp_logistica,
    "VN_nao_alfabetizado": vn_logistica
})

### 6.3 Comparação com Regressão Logística sem balanceamento

A Regressão Logística balanceada apresentou aumento expressivo na capacidade de identificar alunos não alfabetizados em relação ao baseline ingênuo, porém acompanhado por quantidade relevante de falsos positivos.

Para avaliar especificamente o efeito da ponderação das classes, será treinada uma segunda Regressão Logística sem `class_weight="balanced"`.

As demais configurações serão mantidas iguais, incluindo as features, o conjunto de treino, o pré-processamento com padronização e os parâmetros do estimador.

Dessa forma, a comparação entre as duas versões permitirá observar empiricamente o efeito da ponderação das classes sobre recall, precisão, F1-score e demais métricas de validação.

In [0]:
# Objetivo:
#
# Construir e treinar uma Regressão Logística
# sem balanceamento automático das classes.
#
# Justificativa:
#
# A comparação com a versão balanceada permitirá
# avaliar o efeito específico de class_weight
# sobre o comportamento do modelo.
#
# Para tornar a comparação consistente, todas
# as demais configurações serão mantidas iguais.
#
# Ação:
#
# Constrói uma nova pipeline com o mesmo
# pré-processamento utilizado anteriormente,
# mas sem ponderação das classes, e realiza
# o ajuste exclusivamente no conjunto de treino.

pipeline_logistica_sem_balanceamento = Pipeline(
    steps=[
        (
            "pre_processamento",
            criar_pre_processamento(
                padronizar=True
            )
        ),
        (
            "modelo",
            LogisticRegression(
                class_weight=None,
                max_iter=1000,
                random_state=42
            )
        )
    ]
)


pipeline_logistica_sem_balanceamento.fit(
    X_treino,
    y_treino
)

### 6.4 Avaliação da Regressão Logística sem balanceamento

Após o treinamento, a Regressão Logística sem balanceamento será avaliada no mesmo conjunto de validação e com as mesmas métricas utilizadas na versão balanceada.

Como todas as demais configurações foram mantidas, essa comparação permitirá observar o efeito da ponderação das classes sobre o comportamento do modelo.

A análise continuará priorizando a classe `0` — não alfabetizado — com especial atenção ao recall, à precisão e ao F1-score.

O conjunto de teste permanecerá isolado.

In [0]:
# Objetivo:
#
# Avaliar a Regressão Logística sem
# balanceamento no conjunto de validação.
#
# Justificativa:
#
# A utilização das mesmas métricas e do mesmo
# conjunto de validação permite comparar esta
# configuração diretamente com a versão
# balanceada.
#
# Como a classe positiva de interesse é 0,
# recall, precisão e F1 serão calculados
# explicitamente com pos_label=0.
#
# Ação:
#
# Realiza as previsões no conjunto de validação
# e calcula as mesmas métricas utilizadas na
# avaliação da Regressão Logística balanceada.

y_pred_logistica_sem_balanceamento = (
    pipeline_logistica_sem_balanceamento.predict(
        X_validacao
    )
)


metricas_logistica_sem_balanceamento = pd.Series({
    "accuracy": accuracy_score(
        y_validacao,
        y_pred_logistica_sem_balanceamento
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_validacao,
        y_pred_logistica_sem_balanceamento
    ),
    "recall_nao_alfabetizado": recall_score(
        y_validacao,
        y_pred_logistica_sem_balanceamento,
        pos_label=0
    ),
    "precision_nao_alfabetizado": precision_score(
        y_validacao,
        y_pred_logistica_sem_balanceamento,
        pos_label=0,
        zero_division=0
    ),
    "f1_nao_alfabetizado": f1_score(
        y_validacao,
        y_pred_logistica_sem_balanceamento,
        pos_label=0,
        zero_division=0
    )
})


metricas_logistica_sem_balanceamento

### 6.5 Matriz de confusão da Regressão Logística sem balanceamento

A matriz de confusão será utilizada para complementar a comparação entre as duas configurações da Regressão Logística.

A análise permitirá quantificar o efeito da retirada do balanceamento das classes sobre verdadeiros positivos, falsos negativos, falsos positivos e verdadeiros negativos.

Como todas as demais condições do experimento foram mantidas, a comparação com a versão balanceada permitirá compreender de forma concreta o trade-off provocado pela ponderação das classes.

A interpretação continuará considerando `0` — não alfabetizado — como classe positiva de interesse.

In [0]:
# Objetivo:
#
# Construir e interpretar a matriz de confusão
# da Regressão Logística sem balanceamento.
#
# Justificativa:
#
# A comparação com a matriz da versão balanceada
# permitirá observar em quantidades de alunos
# como a ponderação das classes afetou os erros
# e acertos do modelo.
#
# A mesma convenção será mantida, considerando
# a classe 0 como positiva de interesse.
#
# Ação:
#
# Calcula a matriz de confusão no conjunto de
# validação e extrai VP, FN, FP e VN.

matriz_logistica_sem_balanceamento = confusion_matrix(
    y_validacao,
    y_pred_logistica_sem_balanceamento,
    labels=[0, 1]
)


vp_sem_balanceamento = (
    matriz_logistica_sem_balanceamento[0, 0]
)

fn_sem_balanceamento = (
    matriz_logistica_sem_balanceamento[0, 1]
)

fp_sem_balanceamento = (
    matriz_logistica_sem_balanceamento[1, 0]
)

vn_sem_balanceamento = (
    matriz_logistica_sem_balanceamento[1, 1]
)


pd.Series({
    "VP_nao_alfabetizado": vp_sem_balanceamento,
    "FN_nao_alfabetizado": fn_sem_balanceamento,
    "FP_nao_alfabetizado": fp_sem_balanceamento,
    "VN_nao_alfabetizado": vn_sem_balanceamento
})

### 6.6 Comparação das configurações da Regressão Logística

As duas configurações da Regressão Logística apresentaram comportamentos substancialmente diferentes no conjunto de validação.

A versão sem balanceamento apresentou maior acurácia e precisão da classe não alfabetizado, porém identificou apenas uma pequena parcela dos alunos pertencentes à classe de interesse.

A utilização de `class_weight="balanced"` aumentou expressivamente o recall da classe não alfabetizado e reduziu a quantidade de falsos negativos, acompanhada, entretanto, por aumento relevante dos falsos positivos e redução da precisão.

A comparação entre as duas configurações evidencia o trade-off entre identificar uma parcela maior dos alunos não alfabetizados e gerar maior quantidade de alertas incorretos.

Esses resultados não determinam, isoladamente, a escolha do modelo final. A decisão deverá considerar os demais modelos candidatos, o custo dos erros, a estabilidade dos resultados e, posteriormente, a definição do limiar de decisão.

In [0]:
# Objetivo:
#
# Consolidar os resultados dos modelos avaliados
# até esta etapa em uma única tabela.
#
# Justificativa:
#
# A comparação lado a lado facilita a análise
# das diferenças entre o baseline ingênuo e as
# duas configurações da Regressão Logística.
#
# Ação:
#
# Organiza as principais métricas obtidas no
# conjunto de validação em uma tabela comparativa.

comparacao_modelos = pd.DataFrame({
    "DummyClassifier": metricas_dummy,
    "Logistica_balanceada": metricas_logistica_balanceada,
    "Logistica_sem_balanceamento": metricas_logistica_sem_balanceamento
})

comparacao_modelos

## 7. Random Forest

Após a avaliação dos baselines, será construído um `RandomForestClassifier` como primeiro modelo não linear do processo de modelagem.

A Random Forest combina múltiplas árvores de decisão construídas com aleatoriedade nos registros e nas features consideradas durante o treinamento, permitindo capturar relações não lineares e interações entre variáveis.

Como modelos baseados em árvores não dependem da escala das variáveis da mesma forma que modelos lineares, o pré-processamento será utilizado sem padronização das features numéricas.

Nesta primeira configuração serão utilizados parâmetros moderados de complexidade e custo computacional, considerando o elevado volume de registros da base.

Também será utilizada inicialmente a ponderação balanceada das classes, mantendo como prioridade a identificação dos alunos não alfabetizados.

Os valores definidos nesta etapa representam uma configuração inicial de referência e não uma otimização de hiperparâmetros. A busca sistemática por configurações mais adequadas será realizada posteriormente.

In [0]:
# Objetivo:
#
# Construir e treinar a primeira Random Forest
# do processo de modelagem.
#
# Justificativa:
#
# A Random Forest permite capturar relações
# não lineares e interações entre as features,
# oferecendo uma comparação com o baseline
# linear estabelecido anteriormente.
#
# Como modelos baseados em árvores não exigem
# padronização das variáveis, o pré-processamento
# será utilizado com padronizar=False.
#
# A ponderação balanceada será considerada
# inicialmente para aumentar a importância
# relativa da classe menos frequente.
#
# Ação:
#
# Constrói uma pipeline completa contendo
# pré-processamento sem padronização e uma
# Random Forest com configuração inicial
# controlada de complexidade.

from sklearn.ensemble import RandomForestClassifier


pipeline_random_forest_balanceada = Pipeline(
    steps=[
        (
            "pre_processamento",
            criar_pre_processamento(
                padronizar=False
            )
        ),
        (
            "modelo",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=15,
                min_samples_leaf=50,
                max_features="sqrt",
                class_weight="balanced",
                n_jobs=-1,
                random_state=42
            )
        )
    ]
)


pipeline_random_forest_balanceada.fit(
    X_treino,
    y_treino
)

### 7.1 Avaliação da Random Forest balanceada

Após o treinamento, a Random Forest balanceada será avaliada no conjunto de validação utilizando as mesmas métricas aplicadas aos modelos anteriores.

A manutenção do mesmo protocolo de avaliação permite comparar diretamente o comportamento da Random Forest com o baseline ingênuo e com as configurações da Regressão Logística.

Como a classe `0` — não alfabetizado — permanece como classe positiva de interesse, serão priorizados o recall, a precisão e o F1-score dessa classe, além da acurácia e da acurácia balanceada.

O conjunto de teste permanecerá isolado nesta etapa.

In [0]:
# Objetivo:
#
# Avaliar a Random Forest balanceada no
# conjunto de validação.
#
# Justificativa:
#
# A utilização das mesmas métricas aplicadas
# aos modelos anteriores permite uma comparação
# consistente entre os diferentes candidatos.
#
# Como a classe positiva de interesse é 0,
# recall, precisão e F1 serão calculados
# explicitamente com pos_label=0.
#
# Ação:
#
# Realiza as previsões no conjunto de validação
# e calcula as principais métricas de avaliação.

y_pred_random_forest_balanceada = (
    pipeline_random_forest_balanceada.predict(
        X_validacao
    )
)


metricas_random_forest_balanceada = pd.Series({
    "accuracy": accuracy_score(
        y_validacao,
        y_pred_random_forest_balanceada
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_validacao,
        y_pred_random_forest_balanceada
    ),
    "recall_nao_alfabetizado": recall_score(
        y_validacao,
        y_pred_random_forest_balanceada,
        pos_label=0
    ),
    "precision_nao_alfabetizado": precision_score(
        y_validacao,
        y_pred_random_forest_balanceada,
        pos_label=0,
        zero_division=0
    ),
    "f1_nao_alfabetizado": f1_score(
        y_validacao,
        y_pred_random_forest_balanceada,
        pos_label=0,
        zero_division=0
    )
})


metricas_random_forest_balanceada

### 7.2 Matriz de confusão da Random Forest balanceada

A matriz de confusão será utilizada para traduzir o desempenho da Random Forest balanceada em quantidades concretas de alunos.

A análise permitirá verificar quantos alunos não alfabetizados foram corretamente identificados e quantos permaneceram como falsos negativos, além de quantificar os falsos positivos e verdadeiros negativos produzidos pelo modelo.

Os resultados também permitirão comparar diretamente o comportamento da Random Forest com o da Regressão Logística balanceada.

A interpretação continuará considerando `0` — não alfabetizado — como classe positiva de interesse.

In [0]:
# Objetivo:
#
# Construir e interpretar a matriz de confusão
# da Random Forest balanceada.
#
# Justificativa:
#
# A matriz de confusão permite traduzir as
# métricas do modelo em quantidades concretas
# de alunos e comparar seu comportamento com
# os modelos avaliados anteriormente.
#
# A mesma convenção será mantida, considerando
# a classe 0 como positiva de interesse.
#
# Ação:
#
# Calcula a matriz de confusão no conjunto de
# validação e extrai VP, FN, FP e VN.

matriz_random_forest_balanceada = confusion_matrix(
    y_validacao,
    y_pred_random_forest_balanceada,
    labels=[0, 1]
)


vp_random_forest = (
    matriz_random_forest_balanceada[0, 0]
)

fn_random_forest = (
    matriz_random_forest_balanceada[0, 1]
)

fp_random_forest = (
    matriz_random_forest_balanceada[1, 0]
)

vn_random_forest = (
    matriz_random_forest_balanceada[1, 1]
)


pd.Series({
    "VP_nao_alfabetizado": vp_random_forest,
    "FN_nao_alfabetizado": fn_random_forest,
    "FP_nao_alfabetizado": fp_random_forest,
    "VN_nao_alfabetizado": vn_random_forest
})

### 7.3 Comparação da Random Forest com os modelos anteriores

Os resultados da primeira configuração da Random Forest serão incorporados à tabela comparativa dos modelos avaliados no conjunto de validação.

A Random Forest balanceada apresentou desempenho muito próximo ao da Regressão Logística balanceada, com pequeno aumento no recall e no F1-score da classe não alfabetizado, acompanhado por pequena redução na precisão.

Na comparação agregada das matrizes de confusão, a Random Forest apresentou 940 verdadeiros positivos adicionais e 940 falsos negativos a menos, ao mesmo tempo em que produziu 1.719 falsos positivos adicionais.

Considerando a proximidade das métricas e o maior custo computacional observado no treinamento da Random Forest, ainda não há evidência suficiente para definir um modelo vencedor.

A comparação será ampliada com os demais modelos candidatos antes da seleção e otimização do modelo final.

In [0]:
# Objetivo:
#
# Incorporar os resultados da Random Forest
# balanceada à tabela comparativa dos modelos.
#
# Justificativa:
#
# A manutenção de uma tabela consolidada permite
# acompanhar a evolução dos candidatos utilizando
# o mesmo conjunto e protocolo de validação.
#
# Ação:
#
# Adiciona as métricas da Random Forest
# balanceada ao DataFrame de comparação
# construído anteriormente.

comparacao_modelos[
    "Random_Forest_balanceada"
] = metricas_random_forest_balanceada


comparacao_modelos

## 8. HistGradientBoostingClassifier

Após a avaliação do baseline ingênuo, da Regressão Logística e da Random Forest, a próxima etapa da modelagem será dedicada ao `HistGradientBoostingClassifier`.

O novo candidato deverá seguir o mesmo protocolo metodológico adotado até aqui:

- utilizar exclusivamente o conjunto de treino durante o ajuste;
- realizar a comparação no conjunto de validação;
- manter o conjunto de teste isolado;
- avaliar as mesmas métricas utilizadas nos modelos anteriores;
- analisar a matriz de confusão considerando a classe `0` — não alfabetizado — como classe positiva de interesse;
- incorporar posteriormente seus resultados à tabela consolidada de comparação dos modelos.

A construção desta etapa será realizada dando continuidade ao processo de modelagem desenvolvido neste notebook.

In [0]:
# Objetivo:
#
# Construir e treinar o modelo
# HistGradientBoostingClassifier.
#
# Justificativa:
#
# O HistGradientBoostingClassifier é um modelo
# de boosting baseado em histogramas, eficiente
# para bases extensas e capaz de representar
# relações não lineares e interações.
#
# Esse estimador exige uma matriz densa. Por
# esse motivo, o pré-processamento será criado
# com saida_densa=True, sem alterar a configuração
# utilizada pelos modelos anteriores.
#
# A ponderação das classes será aplicada por
# meio de sample_weight, preservando a prioridade
# atribuída à classe menos frequente e mantendo
# compatibilidade entre versões do Scikit-learn.
#
# Ação:
#
# Calcula os pesos balanceados, constrói a
# pipeline completa e realiza o ajuste somente
# sobre o conjunto de treino.

from sklearn.ensemble import (
    HistGradientBoostingClassifier
)
from sklearn.utils.class_weight import (
    compute_sample_weight
)


pesos_treino_hist_gradient_boosting = (
    compute_sample_weight(
        class_weight="balanced",
        y=y_treino
    )
)


pipeline_hist_gradient_boosting = Pipeline(
    steps=[
        (
            "pre_processamento",
            criar_pre_processamento(
                padronizar=False,
                saida_densa=True
            )
        ),
        (
            "modelo",
            HistGradientBoostingClassifier(
                loss="log_loss",
                learning_rate=0.08,
                max_iter=150,
                max_leaf_nodes=31,
                min_samples_leaf=100,
                l2_regularization=1.0,
                early_stopping=True,
                validation_fraction=0.10,
                n_iter_no_change=10,
                random_state=42
            )
        )
    ]
)


pipeline_hist_gradient_boosting.fit(
    X_treino,
    y_treino,
    modelo__sample_weight=(
        pesos_treino_hist_gradient_boosting
    )
)


pd.Series({
    "iteracoes_executadas": (
        pipeline_hist_gradient_boosting
        .named_steps["modelo"]
        .n_iter_
    ),
    "early_stopping_ativo": (
        pipeline_hist_gradient_boosting
        .named_steps["modelo"]
        .early_stopping
    )
})

### 8.1 Avaliação do HistGradientBoostingClassifier

Após o treinamento, o novo modelo será avaliado no mesmo conjunto de validação e com as mesmas métricas utilizadas nos candidatos anteriores.

A manutenção do protocolo permite comparar o desempenho sem utilizar o conjunto de teste e sem alterar as partições territoriais definidas no Notebook 07.

Como a classe `0` — não alfabetizado — permanece como classe positiva de interesse, recall, precisão e F1-score serão calculados explicitamente sob essa perspectiva.

In [0]:
# Objetivo:
#
# Avaliar o HistGradientBoostingClassifier
# no conjunto de validação.
#
# Justificativa:
#
# O uso do mesmo conjunto e das mesmas métricas
# permite comparar o novo modelo de forma
# consistente com os candidatos anteriores.
#
# Como a classe positiva de interesse é 0,
# recall, precisão e F1 serão calculados
# explicitamente com pos_label=0.
#
# Ação:
#
# Realiza as previsões no conjunto de validação
# e calcula as principais métricas.

y_pred_hist_gradient_boosting = (
    pipeline_hist_gradient_boosting.predict(
        X_validacao
    )
)


metricas_hist_gradient_boosting = pd.Series({
    "accuracy": accuracy_score(
        y_validacao,
        y_pred_hist_gradient_boosting
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_validacao,
        y_pred_hist_gradient_boosting
    ),
    "recall_nao_alfabetizado": recall_score(
        y_validacao,
        y_pred_hist_gradient_boosting,
        pos_label=0
    ),
    "precision_nao_alfabetizado": precision_score(
        y_validacao,
        y_pred_hist_gradient_boosting,
        pos_label=0,
        zero_division=0
    ),
    "f1_nao_alfabetizado": f1_score(
        y_validacao,
        y_pred_hist_gradient_boosting,
        pos_label=0,
        zero_division=0
    )
})


metricas_hist_gradient_boosting

### 8.2 Matriz de confusão do HistGradientBoostingClassifier

A matriz de confusão traduzirá o desempenho do modelo em quantidades concretas de alunos e permitirá comparar verdadeiros positivos, falsos negativos, falsos positivos e verdadeiros negativos com os demais candidatos.

A interpretação continuará considerando `0` — não alfabetizado — como classe positiva de interesse. O falso negativo permanece como erro prioritário, pois representa um aluno não alfabetizado que deixou de ser identificado.

In [0]:
# Objetivo:
#
# Construir e interpretar a matriz de confusão
# do HistGradientBoostingClassifier.
#
# Justificativa:
#
# A matriz permite observar em números absolutos
# os acertos e erros do novo modelo e comparar
# seu comportamento com os demais candidatos.
#
# Ação:
#
# Calcula a matriz com ordem explícita das
# classes e extrai VP, FN, FP e VN.

matriz_hist_gradient_boosting = confusion_matrix(
    y_validacao,
    y_pred_hist_gradient_boosting,
    labels=[0, 1]
)


vp_hist_gradient_boosting = (
    matriz_hist_gradient_boosting[0, 0]
)

fn_hist_gradient_boosting = (
    matriz_hist_gradient_boosting[0, 1]
)

fp_hist_gradient_boosting = (
    matriz_hist_gradient_boosting[1, 0]
)

vn_hist_gradient_boosting = (
    matriz_hist_gradient_boosting[1, 1]
)


pd.Series({
    "VP_nao_alfabetizado": (
        vp_hist_gradient_boosting
    ),
    "FN_nao_alfabetizado": (
        fn_hist_gradient_boosting
    ),
    "FP_nao_alfabetizado": (
        fp_hist_gradient_boosting
    ),
    "VN_nao_alfabetizado": (
        vn_hist_gradient_boosting
    )
})

### 8.3 Comparação com os modelos anteriores

As métricas do `HistGradientBoostingClassifier` serão adicionadas à tabela consolidada construída ao longo do notebook.

Esta comparação ainda representa a configuração inicial dos modelos. Portanto, não deverá ser utilizada isoladamente para declarar um vencedor. A seleção posterior deverá considerar o equilíbrio entre recall e precisão da classe não alfabetizado, o F1-score, a acurácia balanceada, a matriz de confusão, a estabilidade territorial e o custo computacional.

In [0]:
# Objetivo:
#
# Incorporar o HistGradientBoostingClassifier
# à comparação consolidada dos modelos.
#
# Justificativa:
#
# A tabela lado a lado permite verificar o
# comportamento de todos os candidatos sob
# o mesmo protocolo de validação.
#
# Ação:
#
# Adiciona as métricas do novo modelo e ordena
# as colunas conforme a sequência de construção.

comparacao_modelos[
    "Hist_Gradient_Boosting_balanceado"
] = metricas_hist_gradient_boosting


comparacao_modelos

### 8.4 Encerramento da construção inicial dos modelos candidatos

Com a inclusão do `HistGradientBoostingClassifier`, o notebook passa a contemplar os quatro modelos candidatos previstos para esta fase inicial:

- `DummyClassifier`, como baseline mínimo;
- `LogisticRegression`, como baseline explicável, com e sem balanceamento;
- `RandomForestClassifier`, como modelo não linear baseado em combinação de árvores;
- `HistGradientBoostingClassifier`, como modelo de boosting eficiente para bases extensas.

Todos os candidatos foram integrados ao mesmo módulo de pré-processamento, treinados exclusivamente sobre o conjunto de treino e avaliados sobre o conjunto de validação. O conjunto de teste permanece isolado.

A configuração do HistGradientBoosting apresentada nesta seção constitui um ponto inicial controlado, e não uma otimização final. As próximas etapas deverão aprofundar a comparação, a validação estatística, a seleção de hiperparâmetros e a definição do limiar de decisão antes da avaliação final no conjunto de teste.

## 9. Desbalanceamento e limiar de decisão

A análise inicial confirmou que o target é desbalanceado e que a classe `0` — aluno não alfabetizado — representa a classe minoritária e a condição de interesse do projeto.

Os modelos balanceados já utilizaram `class_weight="balanced"` ou pesos de amostra. Nesta etapa, o objetivo não será reamostrar fisicamente os dados, pois isso aumentaria o custo computacional e poderia comprometer a separação territorial. A avaliação será aprofundada por meio das probabilidades previstas para a classe de risco.

O limiar padrão de `0,50` não será considerado automaticamente como o ponto ideal. Será analisado o equilíbrio entre recall, precisão, F1-score e volume de alertas, utilizando exclusivamente o conjunto de validação. O conjunto de teste permanecerá isolado.

In [0]:
# Objetivo:
#
# Consolidar o diagnóstico de desbalanceamento
# e preparar o diretório de resultados.
#
# Justificativa:
#
# A proporção das classes orienta a escolha das
# métricas, o uso de ponderação e a interpretação
# dos limiares de decisão.
#
# Os artefatos analíticos desta fase devem ser
# gravados somente na estrutura da Fase 3.
#
# Ação:
#
# Calcula o perfil do target de treino e cria
# o diretório de resultados da modelagem.

import os


RESULTADOS_PATH = (
    f"{MODELAGEM_PATH}/resultados"
)

dbutils.fs.mkdirs(RESULTADOS_PATH)


diagnostico_desbalanceamento = pd.DataFrame({
    "quantidade": (
        y_treino.value_counts().sort_index()
    ),
    "percentual": (
        y_treino
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(4)
    )
})

diagnostico_desbalanceamento.index = [
    "0 - Não alfabetizado",
    "1 - Alfabetizado"
]


diagnostico_desbalanceamento

### 9.1 Probabilidades e capacidade de discriminação

As classes previstas dependem do limiar, enquanto as probabilidades permitem avaliar a capacidade de ordenação dos alunos por risco.

Para manter a interpretação alinhada ao problema, será extraída explicitamente a probabilidade da classe `0`. A função não presume a posição dessa classe no retorno de `predict_proba`; ela consulta `classes_` em cada estimador.

Serão calculadas:

- PR-AUC da classe não alfabetizado, métrica principal desta comparação;
- ROC-AUC, como medida complementar de discriminação;
- Brier Score, como diagnóstico inicial da qualidade probabilística.

O Brier Score será apenas diagnóstico nesta etapa. A calibração formal será tratada na avaliação estatística posterior.

In [0]:
# Objetivo:
#
# Extrair a probabilidade da classe 0 e comparar
# a discriminação dos modelos iniciais.
#
# Justificativa:
#
# O predict_proba segue a ordem de classes_ do
# estimador. Consultar essa ordem evita utilizar
# acidentalmente a probabilidade da classe 1.
#
# A PR-AUC é adequada ao foco na classe
# minoritária, enquanto ROC-AUC e Brier Score
# complementam o diagnóstico.
#
# Ação:
#
# Calcula as probabilidades no conjunto de
# validação e consolida as métricas.

import numpy as np

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score
)


def probabilidade_classe(
    pipeline,
    X,
    classe_interesse=0
):
    modelo = pipeline.named_steps["modelo"]
    classes = np.asarray(modelo.classes_)

    posicoes = np.flatnonzero(
        classes == classe_interesse
    )

    if len(posicoes) != 1:
        raise ValueError(
            "Classe de interesse não encontrada "
            "de forma única no estimador."
        )

    return pipeline.predict_proba(X)[
        :,
        int(posicoes[0])
    ]


modelos_iniciais = {
    "DummyClassifier": pipeline_dummy,
    "Logistica_balanceada": (
        pipeline_logistica_balanceada
    ),
    "Logistica_sem_balanceamento": (
        pipeline_logistica_sem_balanceamento
    ),
    "Random_Forest_balanceada": (
        pipeline_random_forest_balanceada
    ),
    "Hist_Gradient_Boosting_balanceado": (
        pipeline_hist_gradient_boosting
    )
}


y_validacao_risco = (
    y_validacao.eq(0).astype("int8")
)


probabilidades_validacao = {}
linhas_discriminacao = []


for nome_modelo, pipeline_modelo in (
    modelos_iniciais.items()
):
    probabilidades = probabilidade_classe(
        pipeline_modelo,
        X_validacao,
        classe_interesse=0
    )

    probabilidades_validacao[nome_modelo] = (
        probabilidades
    )

    linhas_discriminacao.append({
        "modelo": nome_modelo,
        "pr_auc_risco": average_precision_score(
            y_validacao_risco,
            probabilidades
        ),
        "roc_auc_risco": roc_auc_score(
            y_validacao_risco,
            probabilidades
        ),
        "brier_score_risco": brier_score_loss(
            y_validacao_risco,
            probabilidades
        )
    })


comparacao_discriminacao_inicial = (
    pd.DataFrame(linhas_discriminacao)
    .set_index("modelo")
    .sort_values(
        by="pr_auc_risco",
        ascending=False
    )
)


comparacao_discriminacao_inicial

### 9.2 Curvas Precision-Recall e ROC

A curva Precision-Recall evidencia o trade-off entre a capacidade de encontrar alunos não alfabetizados e a qualidade dos alertas emitidos. Ela será priorizada porque a classe de risco é minoritária.

A curva ROC será apresentada como complemento. Um modelo pode apresentar ROC-AUC aparentemente satisfatória e ainda possuir precisão limitada na classe minoritária; por isso, as duas curvas não devem ser interpretadas isoladamente.

In [0]:
# Objetivo:
#
# Visualizar a discriminação dos modelos ao
# longo de diferentes limiares.
#
# Justificativa:
#
# As curvas permitem comparar os modelos sem
# fixar previamente um único limiar de decisão.
#
# Ação:
#
# Constrói as curvas Precision-Recall e ROC
# para a classe não alfabetizado.

import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_recall_curve,
    roc_curve
)


fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 5)
)


for nome_modelo, probabilidades in (
    probabilidades_validacao.items()
):
    precisao_curva, recall_curva, _ = (
        precision_recall_curve(
            y_validacao_risco,
            probabilidades
        )
    )

    falso_positivo, verdadeiro_positivo, _ = (
        roc_curve(
            y_validacao_risco,
            probabilidades
        )
    )

    axes[0].plot(
        recall_curva,
        precisao_curva,
        label=nome_modelo
    )

    axes[1].plot(
        falso_positivo,
        verdadeiro_positivo,
        label=nome_modelo
    )


prevalencia_risco = y_validacao_risco.mean()

axes[0].axhline(
    prevalencia_risco,
    color="gray",
    linestyle="--",
    label="Prevalência da classe de risco"
)

axes[0].set_title(
    "Curva Precision-Recall — classe 0"
)
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precisão")
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=8)


axes[1].plot(
    [0, 1],
    [0, 1],
    color="gray",
    linestyle="--",
    label="Referência aleatória"
)
axes[1].set_title("Curva ROC — classe 0")
axes[1].set_xlabel("Taxa de falsos positivos")
axes[1].set_ylabel("Taxa de verdadeiros positivos")
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=8)


plt.tight_layout()
plt.show()

### 9.3 Seleção provisória do modelo de referência e do limiar

O modelo inicial com maior PR-AUC será utilizado como referência para a análise de limiar. Essa escolha avalia a capacidade de ordenar alunos por risco antes da conversão das probabilidades em classes.

Como regra inicial de negócio, será buscado recall mínimo de 70% para os alunos não alfabetizados. Entre os limiares que atendam a essa cobertura, será escolhido aquele com maior precisão; em caso de empate, prevalecerá o maior F1-score. Se nenhum limiar atender ao recall mínimo, será utilizado o maior F1-score.

O valor de 70% é um critério técnico provisório e deverá ser validado com os stakeholders. O limiar desta seção também é provisório, pois será recalculado após a otimização.

In [0]:
# Objetivo:
#
# Selecionar um modelo inicial de referência
# e analisar diferentes limiares.
#
# Justificativa:
#
# O limiar 0,50 pode não representar o melhor
# equilíbrio para a classe de risco.
#
# A regra prioriza cobertura mínima dos alunos
# não alfabetizados e, depois, a qualidade dos
# alertas gerados.
#
# Ação:
#
# Avalia limiares de 0,05 a 0,95 e seleciona
# provisoriamente o ponto de operação.


def avaliar_limiares(
    y_real,
    probabilidade_risco,
    limiares
):
    registros = []

    y_real_array = np.asarray(y_real)
    risco_real = y_real_array == 0

    for limiar in limiares:
        risco_previsto = (
            probabilidade_risco >= limiar
        )

        vp = np.sum(risco_real & risco_previsto)
        fn = np.sum(risco_real & ~risco_previsto)
        fp = np.sum(~risco_real & risco_previsto)
        vn = np.sum(~risco_real & ~risco_previsto)

        recall = (
            vp / (vp + fn)
            if (vp + fn) > 0
            else 0.0
        )

        precisao = (
            vp / (vp + fp)
            if (vp + fp) > 0
            else 0.0
        )

        f1 = (
            2 * precisao * recall
            / (precisao + recall)
            if (precisao + recall) > 0
            else 0.0
        )

        registros.append({
            "limiar": float(limiar),
            "recall_nao_alfabetizado": recall,
            "precision_nao_alfabetizado": precisao,
            "f1_nao_alfabetizado": f1,
            "alertas_gerados": int(vp + fp),
            "percentual_alertas": (
                (vp + fp) / len(y_real_array)
            ),
            "VP": int(vp),
            "FN": int(fn),
            "FP": int(fp),
            "VN": int(vn)
        })

    return pd.DataFrame(registros)


def selecionar_limiar(
    tabela_limiares,
    recall_minimo=0.70
):
    candidatos = tabela_limiares.loc[
        tabela_limiares[
            "recall_nao_alfabetizado"
        ] >= recall_minimo
    ]

    if not candidatos.empty:
        linha = candidatos.sort_values(
            by=[
                "precision_nao_alfabetizado",
                "f1_nao_alfabetizado",
                "limiar"
            ],
            ascending=[False, False, False]
        ).iloc[0]

        criterio = (
            "maior precisão com recall mínimo "
            f"de {recall_minimo:.0%}"
        )
    else:
        linha = tabela_limiares.sort_values(
            by=[
                "f1_nao_alfabetizado",
                "recall_nao_alfabetizado"
            ],
            ascending=[False, False]
        ).iloc[0]

        criterio = (
            "maior F1; nenhum limiar atingiu "
            "o recall mínimo"
        )

    return linha, criterio


nome_modelo_referencia_inicial = (
    comparacao_discriminacao_inicial.index[0]
)

pipeline_referencia_inicial = modelos_iniciais[
    nome_modelo_referencia_inicial
]

probabilidade_referencia_inicial = (
    probabilidades_validacao[
        nome_modelo_referencia_inicial
    ]
)


RECALL_MINIMO = 0.70

limiares_avaliados = np.round(
    np.arange(0.05, 0.951, 0.01),
    2
)

analise_limiares_inicial = avaliar_limiares(
    y_validacao,
    probabilidade_referencia_inicial,
    limiares_avaliados
)

limiar_inicial_selecionado, criterio_inicial = (
    selecionar_limiar(
        analise_limiares_inicial,
        recall_minimo=RECALL_MINIMO
    )
)


pd.Series({
    "modelo_referencia_inicial": (
        nome_modelo_referencia_inicial
    ),
    "limiar_provisorio": (
        limiar_inicial_selecionado["limiar"]
    ),
    "criterio": criterio_inicial,
    "recall_nao_alfabetizado": (
        limiar_inicial_selecionado[
            "recall_nao_alfabetizado"
        ]
    ),
    "precision_nao_alfabetizado": (
        limiar_inicial_selecionado[
            "precision_nao_alfabetizado"
        ]
    ),
    "f1_nao_alfabetizado": (
        limiar_inicial_selecionado[
            "f1_nao_alfabetizado"
        ]
    ),
    "percentual_alertas": (
        limiar_inicial_selecionado[
            "percentual_alertas"
        ]
    )
})

### 9.4 Curvas de métricas por limiar

O gráfico a seguir mostra como recall, precisão, F1-score e volume de alertas variam conforme o limiar. O ponto selecionado será destacado para facilitar a interpretação e a futura discussão com gestores educacionais.

In [0]:
# Objetivo:
#
# Visualizar o efeito do limiar nas métricas
# e no volume de alertas.
#
# Justificativa:
#
# A escolha do limiar representa uma decisão
# operacional e precisa ser interpretável.
#
# Ação:
#
# Exibe as métricas por limiar e destaca o
# ponto provisoriamente selecionado.

limiar_provisorio = float(
    limiar_inicial_selecionado["limiar"]
)


fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

axes[0].plot(
    analise_limiares_inicial["limiar"],
    analise_limiares_inicial[
        "recall_nao_alfabetizado"
    ],
    label="Recall"
)
axes[0].plot(
    analise_limiares_inicial["limiar"],
    analise_limiares_inicial[
        "precision_nao_alfabetizado"
    ],
    label="Precisão"
)
axes[0].plot(
    analise_limiares_inicial["limiar"],
    analise_limiares_inicial[
        "f1_nao_alfabetizado"
    ],
    label="F1-score"
)
axes[0].axvline(
    limiar_provisorio,
    color="black",
    linestyle="--",
    label=f"Limiar {limiar_provisorio:.2f}"
)
axes[0].axhline(
    RECALL_MINIMO,
    color="gray",
    linestyle=":",
    label=f"Recall mínimo {RECALL_MINIMO:.0%}"
)
axes[0].set_title(
    "Métricas da classe 0 por limiar"
)
axes[0].set_xlabel("Limiar")
axes[0].set_ylabel("Métrica")
axes[0].grid(alpha=0.25)
axes[0].legend()


axes[1].plot(
    analise_limiares_inicial["limiar"],
    analise_limiares_inicial[
        "percentual_alertas"
    ].mul(100),
    color="darkorange"
)
axes[1].axvline(
    limiar_provisorio,
    color="black",
    linestyle="--"
)
axes[1].set_title(
    "Percentual de alunos sinalizados"
)
axes[1].set_xlabel("Limiar")
axes[1].set_ylabel("Alertas (%)")
axes[1].grid(alpha=0.25)


plt.tight_layout()
plt.show()

### 9.5 Matriz de confusão no limiar provisório

As probabilidades do modelo de referência serão convertidas em classes utilizando o limiar provisório. A matriz resultante permitirá comparar o novo ponto de operação com o comportamento observado no limiar padrão.

Nenhuma decisão será aplicada ao conjunto de teste nesta etapa.

In [0]:
# Objetivo:
#
# Avaliar o modelo inicial de referência no
# limiar provisoriamente selecionado.
#
# Justificativa:
#
# A matriz de confusão traduz a escolha do
# limiar em alunos corretamente identificados
# e erros produzidos.
#
# Ação:
#
# Converte as probabilidades em classes e
# consolida a avaliação provisória.

y_pred_referencia_limiar_inicial = np.where(
    probabilidade_referencia_inicial
    >= limiar_provisorio,
    0,
    1
)

matriz_referencia_limiar_inicial = (
    confusion_matrix(
        y_validacao,
        y_pred_referencia_limiar_inicial,
        labels=[0, 1]
    )
)


metricas_referencia_limiar_inicial = pd.Series({
    "modelo": nome_modelo_referencia_inicial,
    "limiar": limiar_provisorio,
    "accuracy": accuracy_score(
        y_validacao,
        y_pred_referencia_limiar_inicial
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_validacao,
        y_pred_referencia_limiar_inicial
    ),
    "recall_nao_alfabetizado": recall_score(
        y_validacao,
        y_pred_referencia_limiar_inicial,
        pos_label=0
    ),
    "precision_nao_alfabetizado": precision_score(
        y_validacao,
        y_pred_referencia_limiar_inicial,
        pos_label=0,
        zero_division=0
    ),
    "f1_nao_alfabetizado": f1_score(
        y_validacao,
        y_pred_referencia_limiar_inicial,
        pos_label=0,
        zero_division=0
    ),
    "VP": matriz_referencia_limiar_inicial[0, 0],
    "FN": matriz_referencia_limiar_inicial[0, 1],
    "FP": matriz_referencia_limiar_inicial[1, 0],
    "VN": matriz_referencia_limiar_inicial[1, 1]
})


metricas_referencia_limiar_inicial

### 9.6 Conclusão provisória sobre desbalanceamento e limiar

O desbalanceamento foi tratado inicialmente por ponderação, sem alterar fisicamente a distribuição dos registros. A análise probabilística permite verificar se o baixo desempenho de determinado modelo decorre de sua capacidade de discriminação ou apenas do limiar padrão.

O modelo e o limiar selecionados nesta seção são referências provisórias. Após a otimização de hiperparâmetros, as probabilidades serão recalculadas e o limiar será novamente definido antes de qualquer uso do conjunto de teste.

## 10. Otimização de hiperparâmetros

A otimização será executada sobre uma amostra territorial do conjunto de treino, sem utilizar validação ou teste no ajuste dos hiperparâmetros.

Serão avaliadas as três famílias de modelos capazes de aprender relações preditivas:

- Regressão Logística balanceada;
- Random Forest balanceada;
- HistGradientBoosting balanceado.

O DummyClassifier não será otimizado porque sua função é exclusivamente fornecer o baseline mínimo.

A busca utilizará `RandomizedSearchCV` com `StratifiedGroupKFold`, mantendo todos os registros de um mesmo município na mesma dobra. A PR-AUC da classe `0` será a métrica principal de seleção.

### 10.1 Amostra territorial para otimização

Executar dezenas de ajustes sobre mais de 1,3 milhão de registros pode elevar significativamente o tempo e a memória consumidos. Para controlar esse custo, será selecionada uma fração dos municípios do treino exclusivamente para a busca de hiperparâmetros.

A amostra mantém municípios inteiros. Após a seleção dos melhores parâmetros, cada pipeline será reajustada sobre o conjunto completo de treino.

In [0]:
# Objetivo:
#
# Criar uma amostra territorial do treino para
# a busca de hiperparâmetros.
#
# Justificativa:
#
# A seleção de municípios inteiros reduz o custo
# sem misturar o mesmo território entre as
# partições internas da otimização.
#
# Ação:
#
# Seleciona 25% dos municípios do conjunto de
# treino e valida a amostra resultante.

from sklearn.model_selection import (
    GroupShuffleSplit
)


FRACAO_MUNICIPIOS_TUNING = 0.25

grupos_treino = (
    auxiliares_treino["co_municipio"]
    .astype("string")
    .fillna("SEM_MUNICIPIO")
)


seletor_tuning = GroupShuffleSplit(
    n_splits=1,
    train_size=FRACAO_MUNICIPIOS_TUNING,
    random_state=42
)

indices_tuning, _ = next(
    seletor_tuning.split(
        X_treino,
        y_treino,
        groups=grupos_treino
    )
)


X_tuning = (
    X_treino.iloc[indices_tuning]
    .reset_index(drop=True)
)

y_tuning = (
    y_treino.iloc[indices_tuning]
    .reset_index(drop=True)
)

grupos_tuning = (
    grupos_treino.iloc[indices_tuning]
    .reset_index(drop=True)
)


pd.Series({
    "registros_treino_completo": len(X_treino),
    "registros_amostra_tuning": len(X_tuning),
    "municipios_treino_completo": (
        grupos_treino.nunique()
    ),
    "municipios_amostra_tuning": (
        grupos_tuning.nunique()
    ),
    "percentual_classe_0_treino": (
        y_treino.eq(0).mean() * 100
    ),
    "percentual_classe_0_tuning": (
        y_tuning.eq(0).mean() * 100
    )
})

### 10.2 Validação cruzada agrupada e métrica principal

A validação cruzada utilizará três dobras estratificadas e agrupadas por município. A estratificação busca preservar o target, enquanto o agrupamento impede que um mesmo município apareça simultaneamente no treino e na avaliação interna de uma dobra.

Será utilizada uma função de scoring específica para PR-AUC da classe `0`, extraindo explicitamente sua probabilidade.

In [0]:
# Objetivo:
#
# Configurar a validação cruzada territorial
# e o scorer da classe de risco.
#
# Justificativa:
#
# O agrupamento por município reduz o otimismo
# provocado pela repetição de features contextuais.
#
# Ação:
#
# Cria o StratifiedGroupKFold e uma função de
# PR-AUC explicitamente orientada à classe 0.

from sklearn.model_selection import (
    StratifiedGroupKFold
)


def scorer_pr_auc_classe_zero(
    estimador,
    X,
    y
):
    probabilidades = probabilidade_classe(
        estimador,
        X,
        classe_interesse=0
    )

    y_risco = (
        pd.Series(y)
        .reset_index(drop=True)
        .eq(0)
        .astype("int8")
    )

    return average_precision_score(
        y_risco,
        probabilidades
    )


cv_territorial = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)


scoring_tuning = {
    "pr_auc_risco": scorer_pr_auc_classe_zero,
    "balanced_accuracy": "balanced_accuracy"
}


print(
    "Validação configurada: 3 dobras "
    "estratificadas e agrupadas por município."
)

### 10.3 Espaços de busca e execução do RandomizedSearchCV

Os espaços de busca foram mantidos intencionalmente compactos para equilibrar qualidade experimental e viabilidade computacional.

Serão realizadas quatro configurações aleatórias por família de modelo e três dobras por configuração. As buscas serão executadas sequencialmente com `n_jobs=1`, evitando que cópias simultâneas das pipelines — principalmente da saída densa do HistGradientBoosting — causem pressão excessiva de memória.

O conjunto de teste e o conjunto de validação não participam do ajuste.

In [0]:
# Objetivo:
#
# Otimizar os hiperparâmetros das três famílias
# de modelos supervisionados.
#
# Justificativa:
#
# O RandomizedSearchCV explora combinações com
# custo controlado e utiliza a pipeline completa,
# impedindo que o pré-processamento seja ajustado
# fora das dobras de treino.
#
# Ação:
#
# Define os espaços de busca e executa as buscas
# sequencialmente sobre a amostra territorial.

from sklearn.model_selection import (
    RandomizedSearchCV
)


N_ITER_TUNING = 4


configuracoes_tuning = {
    "Logistica_balanceada": {
        "pipeline": pipeline_logistica_balanceada,
        "parametros": {
            "modelo__C": [
                0.01,
                0.05,
                0.1,
                0.5,
                1.0,
                5.0
            ],
            "modelo__solver": [
                "lbfgs",
                "saga"
            ]
        }
    },
    "Random_Forest_balanceada": {
        "pipeline": pipeline_random_forest_balanceada,
        "parametros": {
            "modelo__n_estimators": [
                150,
                250,
                350
            ],
            "modelo__max_depth": [
                10,
                15,
                20,
                None
            ],
            "modelo__min_samples_leaf": [
                20,
                50,
                100,
                200
            ],
            "modelo__max_features": [
                "sqrt",
                0.5
            ],
            "modelo__class_weight": [
                "balanced",
                "balanced_subsample"
            ]
        }
    },
    "Hist_Gradient_Boosting_balanceado": {
        "pipeline": pipeline_hist_gradient_boosting,
        "parametros": {
            "modelo__learning_rate": [
                0.03,
                0.05,
                0.08,
                0.10
            ],
            "modelo__max_iter": [
                100,
                150,
                250
            ],
            "modelo__max_leaf_nodes": [
                15,
                31,
                63
            ],
            "modelo__min_samples_leaf": [
                50,
                100,
                200
            ],
            "modelo__l2_regularization": [
                0.0,
                0.5,
                1.0,
                2.0
            ]
        }
    }
}


pesos_tuning_hist_gradient_boosting = (
    compute_sample_weight(
        class_weight="balanced",
        y=y_tuning
    )
)


buscas_tuning = {}


for nome_modelo, configuracao in (
    configuracoes_tuning.items()
):
    print(f"Iniciando tuning: {nome_modelo}")

    busca = RandomizedSearchCV(
        estimator=configuracao["pipeline"],
        param_distributions=(
            configuracao["parametros"]
        ),
        n_iter=N_ITER_TUNING,
        scoring=scoring_tuning,
        refit="pr_auc_risco",
        cv=cv_territorial,
        return_train_score=True,
        n_jobs=1,
        random_state=42,
        verbose=1,
        error_score="raise"
    )

    parametros_fit = {
        "groups": grupos_tuning
    }

    if nome_modelo == (
        "Hist_Gradient_Boosting_balanceado"
    ):
        parametros_fit[
            "modelo__sample_weight"
        ] = pesos_tuning_hist_gradient_boosting

    busca.fit(
        X_tuning,
        y_tuning,
        **parametros_fit
    )

    buscas_tuning[nome_modelo] = busca

    print(
        f"Concluído: {nome_modelo} | "
        f"PR-AUC CV: {busca.best_score_:.6f}"
    )

### 10.4 Resultados da otimização e diagnóstico de overfitting

Para cada família serão registrados o melhor PR-AUC médio, o desvio entre as dobras, a balanced accuracy, o tempo médio de ajuste e a diferença entre PR-AUC de treino e validação cruzada.

Uma diferença elevada entre treino e validação pode indicar overfitting. Baixo desempenho nos dois conjuntos pode sinalizar underfitting ou insuficiência das features disponíveis.

In [0]:
# Objetivo:
#
# Consolidar os resultados das buscas e medir
# estabilidade e possível overfitting.
#
# Justificativa:
#
# A melhor média isolada não é suficiente; é
# necessário considerar dispersão, diferença
# treino-validação e custo computacional.
#
# Ação:
#
# Extrai as informações da melhor configuração
# de cada família de modelos.

linhas_resultados_tuning = []


for nome_modelo, busca in buscas_tuning.items():
    indice = busca.best_index_
    resultados = busca.cv_results_

    pr_auc_treino = resultados[
        "mean_train_pr_auc_risco"
    ][indice]

    pr_auc_cv = resultados[
        "mean_test_pr_auc_risco"
    ][indice]

    linhas_resultados_tuning.append({
        "modelo": nome_modelo,
        "pr_auc_treino": pr_auc_treino,
        "pr_auc_cv": pr_auc_cv,
        "desvio_pr_auc_cv": resultados[
            "std_test_pr_auc_risco"
        ][indice],
        "gap_treino_cv": (
            pr_auc_treino - pr_auc_cv
        ),
        "balanced_accuracy_cv": resultados[
            "mean_test_balanced_accuracy"
        ][indice],
        "tempo_medio_fit_segundos": resultados[
            "mean_fit_time"
        ][indice],
        "melhores_parametros": busca.best_params_
    })


resultados_tuning = (
    pd.DataFrame(linhas_resultados_tuning)
    .set_index("modelo")
    .sort_values(
        by="pr_auc_cv",
        ascending=False
    )
)


resultados_tuning

### 10.5 Reajuste das melhores pipelines no treino completo

As melhores configurações encontradas na amostra territorial serão clonadas e reajustadas sobre todos os registros do conjunto de treino. Nenhum parâmetro será aprendido a partir da validação ou do teste.

Para o HistGradientBoosting, os pesos balanceados serão recalculados sobre o treino completo.

In [0]:
# Objetivo:
#
# Reajustar as melhores pipelines sobre todo o
# conjunto de treino.
#
# Justificativa:
#
# A busca utiliza uma amostra para controlar o
# custo, mas o modelo candidato final deve aprender
# com todo o conjunto de treino disponível.
#
# Ação:
#
# Clona as melhores configurações e realiza o
# ajuste completo, preservando a validação e o teste.

from sklearn.base import clone


pipelines_otimizadas = {}

pesos_treino_hist_otimizado = (
    compute_sample_weight(
        class_weight="balanced",
        y=y_treino
    )
)


for nome_modelo, busca in buscas_tuning.items():
    print(
        "Reajustando no treino completo: "
        f"{nome_modelo}"
    )

    pipeline_otimizada = clone(
        busca.best_estimator_
    )

    if nome_modelo == (
        "Hist_Gradient_Boosting_balanceado"
    ):
        pipeline_otimizada.fit(
            X_treino,
            y_treino,
            modelo__sample_weight=(
                pesos_treino_hist_otimizado
            )
        )
    else:
        pipeline_otimizada.fit(
            X_treino,
            y_treino
        )

    pipelines_otimizadas[nome_modelo] = (
        pipeline_otimizada
    )


print("Reajuste completo concluído.")

### 10.6 Avaliação das pipelines otimizadas na validação

As pipelines otimizadas serão comparadas no conjunto de validação utilizando métricas de classificação no limiar padrão e métricas probabilísticas independentes do limiar.

Essa etapa permite verificar se o ganho observado na validação cruzada se mantém em municípios totalmente separados do conjunto de treino.

In [0]:
# Objetivo:
#
# Avaliar os modelos otimizados no conjunto
# territorial de validação.
#
# Justificativa:
#
# A avaliação externa à validação cruzada mede
# se o ganho se mantém em municípios separados.
#
# Ação:
#
# Calcula métricas de classificação, PR-AUC,
# ROC-AUC e Brier Score para cada pipeline.

linhas_validacao_otimizados = []
probabilidades_otimizadas = {}


for nome_modelo, pipeline_modelo in (
    pipelines_otimizadas.items()
):
    y_pred = pipeline_modelo.predict(
        X_validacao
    )

    probabilidades = probabilidade_classe(
        pipeline_modelo,
        X_validacao,
        classe_interesse=0
    )

    probabilidades_otimizadas[nome_modelo] = (
        probabilidades
    )

    linhas_validacao_otimizados.append({
        "modelo": nome_modelo,
        "accuracy": accuracy_score(
            y_validacao,
            y_pred
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_validacao,
            y_pred
        ),
        "recall_nao_alfabetizado": recall_score(
            y_validacao,
            y_pred,
            pos_label=0
        ),
        "precision_nao_alfabetizado": precision_score(
            y_validacao,
            y_pred,
            pos_label=0,
            zero_division=0
        ),
        "f1_nao_alfabetizado": f1_score(
            y_validacao,
            y_pred,
            pos_label=0,
            zero_division=0
        ),
        "pr_auc_risco": average_precision_score(
            y_validacao_risco,
            probabilidades
        ),
        "roc_auc_risco": roc_auc_score(
            y_validacao_risco,
            probabilidades
        ),
        "brier_score_risco": brier_score_loss(
            y_validacao_risco,
            probabilidades
        )
    })


comparacao_modelos_otimizados = (
    pd.DataFrame(linhas_validacao_otimizados)
    .set_index("modelo")
    .sort_values(
        by="pr_auc_risco",
        ascending=False
    )
)


comparacao_modelos_otimizados

### 10.7 Seleção do modelo otimizado e congelamento do limiar

O modelo otimizado com maior PR-AUC na validação será selecionado como candidato para a etapa seguinte. Seu limiar será recalculado usando a mesma regra definida no item 9: maior precisão entre os pontos que atendem ao recall mínimo de 70%.

Após esta célula, o modelo, seus parâmetros e o limiar serão considerados congelados. O conjunto de teste continuará sem utilização até o item 11.

In [0]:
# Objetivo:
#
# Selecionar o modelo otimizado candidato e
# congelar seu limiar de decisão.
#
# Justificativa:
#
# O limiar precisa ser redefinido após a
# otimização, pois as probabilidades do modelo
# podem ter mudado.
#
# Ação:
#
# Seleciona o maior PR-AUC, avalia os limiares
# e registra a decisão final de desenvolvimento.

nome_modelo_otimizado_selecionado = (
    comparacao_modelos_otimizados.index[0]
)

pipeline_modelo_selecionado = (
    pipelines_otimizadas[
        nome_modelo_otimizado_selecionado
    ]
)

probabilidade_modelo_selecionado = (
    probabilidades_otimizadas[
        nome_modelo_otimizado_selecionado
    ]
)


analise_limiares_otimizado = avaliar_limiares(
    y_validacao,
    probabilidade_modelo_selecionado,
    limiares_avaliados
)

limiar_otimizado_selecionado, criterio_otimizado = (
    selecionar_limiar(
        analise_limiares_otimizado,
        recall_minimo=RECALL_MINIMO
    )
)


LIMIAR_FINAL_VALIDACAO = float(
    limiar_otimizado_selecionado["limiar"]
)


y_pred_modelo_selecionado = np.where(
    probabilidade_modelo_selecionado
    >= LIMIAR_FINAL_VALIDACAO,
    0,
    1
)

matriz_modelo_selecionado = confusion_matrix(
    y_validacao,
    y_pred_modelo_selecionado,
    labels=[0, 1]
)


decisao_modelagem = pd.Series({
    "modelo_selecionado": (
        nome_modelo_otimizado_selecionado
    ),
    "metrica_selecao": "PR-AUC da classe 0",
    "pr_auc_validacao": (
        comparacao_modelos_otimizados.loc[
            nome_modelo_otimizado_selecionado,
            "pr_auc_risco"
        ]
    ),
    "limiar_congelado": LIMIAR_FINAL_VALIDACAO,
    "criterio_limiar": criterio_otimizado,
    "recall_nao_alfabetizado": (
        limiar_otimizado_selecionado[
            "recall_nao_alfabetizado"
        ]
    ),
    "precision_nao_alfabetizado": (
        limiar_otimizado_selecionado[
            "precision_nao_alfabetizado"
        ]
    ),
    "f1_nao_alfabetizado": (
        limiar_otimizado_selecionado[
            "f1_nao_alfabetizado"
        ]
    ),
    "VP": matriz_modelo_selecionado[0, 0],
    "FN": matriz_modelo_selecionado[0, 1],
    "FP": matriz_modelo_selecionado[1, 0],
    "VN": matriz_modelo_selecionado[1, 1],
    "teste_utilizado": False
})


decisao_modelagem

### 10.8 Persistência dos resultados intermediários

As tabelas analíticas e a decisão congelada serão persistidas em `tech_challenge_fase3/modelagem/resultados`. A pipeline treinada ainda não será serializada, pois a validação estatística no conjunto de teste pertence ao item 11 e deve ocorrer antes do empacotamento definitivo.

In [0]:
# Objetivo:
#
# Persistir os resultados dos itens 9 e 10
# na estrutura da Fase 3.
#
# Justificativa:
#
# A persistência garante rastreabilidade das
# decisões sem antecipar o empacotamento final
# do modelo.
#
# Ação:
#
# Salva as tabelas de discriminação, limiares,
# tuning, validação e decisão parcial.

import json


comparacao_discriminacao_inicial.to_csv(
    f"{RESULTADOS_PATH}/"
    "comparacao_discriminacao_inicial.csv",
    sep=";",
    encoding="utf-8-sig"
)

analise_limiares_inicial.to_csv(
    f"{RESULTADOS_PATH}/"
    "analise_limiares_inicial.csv",
    sep=";",
    encoding="utf-8-sig",
    index=False
)

resultados_tuning.to_csv(
    f"{RESULTADOS_PATH}/"
    "resultados_tuning.csv",
    sep=";",
    encoding="utf-8-sig"
)

comparacao_modelos_otimizados.to_csv(
    f"{RESULTADOS_PATH}/"
    "comparacao_modelos_otimizados.csv",
    sep=";",
    encoding="utf-8-sig"
)

analise_limiares_otimizado.to_csv(
    f"{RESULTADOS_PATH}/"
    "analise_limiares_otimizado.csv",
    sep=";",
    encoding="utf-8-sig",
    index=False
)


decisao_serializavel = {
    chave: (
        valor.item()
        if isinstance(valor, np.generic)
        else valor
    )
    for chave, valor in decisao_modelagem.items()
}


with open(
    f"{RESULTADOS_PATH}/"
    "decisao_modelagem_itens_09_10.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        decisao_serializavel,
        arquivo,
        ensure_ascii=False,
        indent=2
    )


pd.Series({
    "diretorio_resultados": RESULTADOS_PATH,
    "arquivos_persistidos": 6,
    "pipeline_final_persistida": False,
    "conjunto_teste_utilizado": False
})

### 10.9 Conclusão da otimização

Os modelos supervisionados foram otimizados com pré-processamento integralmente encapsulado nas pipelines e validação cruzada agrupada por município. Os melhores parâmetros foram reajustados no treino completo e comparados no conjunto territorial de validação.

O modelo candidato e o limiar foram congelados com base na PR-AUC da classe não alfabetizado e na regra de recall mínimo. O conjunto de teste permaneceu isolado durante toda a preparação, comparação, seleção de hiperparâmetros e definição do limiar.

A próxima etapa deverá executar a avaliação final e a validação estatística previstas no item 11, incluindo métricas no teste, calibração, intervalos de confiança agrupados por município, estabilidade por subgrupos e comparação estatística. Somente após essa etapa a pipeline deverá ser empacotada como artefato definitivo.

In [0]:
variaveis_necessarias = [
    "X_teste",
    "y_teste",
    "auxiliares_teste",
    "pipeline_modelo_selecionado",
    "nome_modelo_otimizado_selecionado",
    "LIMIAR_FINAL_VALIDACAO",
    "decisao_modelagem",
    "modelos_iniciais",
    "scorer_pr_auc_classe_zero",
    "probabilidade_classe"
]


status_variaveis = {
    variavel: variavel in globals()
    for variavel in variaveis_necessarias
}


for variavel, existe in status_variaveis.items():
    print(
        f"{variavel}: "
        f"{'OK' if existe else 'AUSENTE'}"
    )


print(
    "\nAmbiente pronto:",
    all(status_variaveis.values())
)

### 11. Avaliação e validação estatística
O modelo, os hiperparâmetros e o limiar foram definidos exclusivamente com os conjuntos de treino e validação. Nesta etapa será realizada a avaliação final no conjunto de teste, composto por municípios que não participaram do desenvolvimento.

O teste será utilizado uma única vez e seus resultados não poderão motivar novo ajuste de hiperparâmetros, troca de features, alteração do limiar ou substituição do modelo. Qualquer melhoria posterior deverá iniciar um novo ciclo experimental com outro conjunto de teste.

Serão avaliadas discriminação, classificação, calibração, incerteza, estabilidade territorial e comparação estatística.

### 11.1 Verificação do congelamento experimental

Antes de acessar o teste, a célula seguinte confirma a existência do modelo selecionado, do limiar congelado e da decisão registrada nos itens anteriores. Também verifica que o teste ainda não foi marcado como utilizado na sessão atual.

In [0]:
# Objetivo:
#
# Validar o congelamento do experimento antes
# da avaliação final no conjunto de teste.
#
# Justificativa:
#
# O teste só pode ser acessado após a definição
# do modelo, dos parâmetros e do limiar.
#
# Ação:
#
# Confirma os artefatos de desenvolvimento e
# impede repetição acidental na mesma sessão.

itens_obrigatorios_teste = {
    "pipeline_modelo_selecionado": (
        "pipeline_modelo_selecionado" in globals()
    ),
    "LIMIAR_FINAL_VALIDACAO": (
        "LIMIAR_FINAL_VALIDACAO" in globals()
    ),
    "decisao_modelagem": (
        "decisao_modelagem" in globals()
    )
}


if not all(itens_obrigatorios_teste.values()):
    itens_ausentes = [
        nome
        for nome, existe in (
            itens_obrigatorios_teste.items()
        )
        if not existe
    ]

    raise RuntimeError(
        "Execute integralmente os itens 9 e 10 "
        "antes da avaliação final. Ausentes: "
        f"{itens_ausentes}"
    )


if globals().get(
    "AVALIACAO_TESTE_EXECUTADA",
    False
):
    raise RuntimeError(
        "A avaliação final já foi executada nesta "
        "sessão. Não repita o teste para ajustar "
        "decisões de modelagem."
    )


pd.Series({
    "modelo_congelado": (
        nome_modelo_otimizado_selecionado
    ),
    "limiar_congelado": (
        LIMIAR_FINAL_VALIDACAO
    ),
    "teste_ainda_isolado": True,
    "registros_teste": len(X_teste),
    "municipios_teste": (
        auxiliares_teste["co_municipio"]
        .nunique(dropna=False)
    )
})

### 11.2 Avaliação final no conjunto de teste

A pipeline congelada produzirá a probabilidade da classe `0` e aplicará exatamente o limiar definido na validação. Serão calculadas métricas de discriminação, classificação e probabilidade.

A variável `AVALIACAO_TESTE_EXECUTADA` será marcada imediatamente após a geração das previsões, impedindo repetição acidental na mesma sessão.

In [0]:
# Objetivo:
#
# Executar a avaliação final e única no teste.
#
# Justificativa:
#
# O teste fornece a estimativa independente de
# generalização para municípios não utilizados
# no desenvolvimento.
#
# Ação:
#
# Gera probabilidades, aplica o limiar congelado
# e calcula as métricas finais.

probabilidade_teste_risco = probabilidade_classe(
    pipeline_modelo_selecionado,
    X_teste,
    classe_interesse=0
)


y_pred_teste_final = np.where(
    probabilidade_teste_risco
    >= LIMIAR_FINAL_VALIDACAO,
    0,
    1
)


AVALIACAO_TESTE_EXECUTADA = True

y_teste_risco = (
    y_teste.eq(0).astype("int8")
)


matriz_confusao_teste_final = confusion_matrix(
    y_teste,
    y_pred_teste_final,
    labels=[0, 1]
)


metricas_teste_final = pd.Series({
    "modelo": nome_modelo_otimizado_selecionado,
    "limiar": LIMIAR_FINAL_VALIDACAO,
    "accuracy": accuracy_score(
        y_teste,
        y_pred_teste_final
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_teste,
        y_pred_teste_final
    ),
    "recall_nao_alfabetizado": recall_score(
        y_teste,
        y_pred_teste_final,
        pos_label=0
    ),
    "precision_nao_alfabetizado": precision_score(
        y_teste,
        y_pred_teste_final,
        pos_label=0,
        zero_division=0
    ),
    "f1_nao_alfabetizado": f1_score(
        y_teste,
        y_pred_teste_final,
        pos_label=0,
        zero_division=0
    ),
    "pr_auc_risco": average_precision_score(
        y_teste_risco,
        probabilidade_teste_risco
    ),
    "roc_auc_risco": roc_auc_score(
        y_teste_risco,
        probabilidade_teste_risco
    ),
    "brier_score_risco": brier_score_loss(
        y_teste_risco,
        probabilidade_teste_risco
    ),
    "VP": matriz_confusao_teste_final[0, 0],
    "FN": matriz_confusao_teste_final[0, 1],
    "FP": matriz_confusao_teste_final[1, 0],
    "VN": matriz_confusao_teste_final[1, 1]
})


decisao_modelagem["teste_utilizado"] = True


metricas_teste_final

### 11.3 Matriz de confusão final

A matriz de confusão apresenta os acertos e erros em números absolutos. A primeira linha corresponde aos alunos realmente não alfabetizados; assim, a célula superior direita representa os falsos negativos prioritários para o projeto.

In [0]:
# Objetivo:
#
# Visualizar a matriz de confusão final.
#
# Justificativa:
#
# A representação gráfica facilita a leitura dos
# falsos negativos e falsos positivos no teste.
#
# Ação:
#
# Exibe a matriz com rótulos de negócio.

from sklearn.metrics import (
    ConfusionMatrixDisplay
)


fig, ax = plt.subplots(figsize=(7, 6))

display_matriz = ConfusionMatrixDisplay(
    confusion_matrix=matriz_confusao_teste_final,
    display_labels=[
        "Não alfabetizado (0)",
        "Alfabetizado (1)"
    ]
)

display_matriz.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

ax.set_title(
    "Matriz de confusão final — conjunto de teste"
)

plt.tight_layout()
plt.show()

### 11.4 Curva de calibração

A calibração verifica se as probabilidades previstas correspondem às frequências observadas. Em um modelo bem calibrado, alunos com risco previsto próximo de 40% pertencem à classe de risco em frequência semelhante.

O Brier Score já calculado resume o erro quadrático probabilístico. A curva por quantis complementará essa avaliação. Como o teste não pode orientar reajustes, eventual necessidade de calibração será registrada como limitação ou como requisito de um novo ciclo experimental.

In [0]:
# Objetivo:
#
# Avaliar visualmente a calibração das
# probabilidades no conjunto de teste.
#
# Justificativa:
#
# Probabilidades utilizadas para ranking ou
# priorização precisam representar riscos
# observáveis de forma coerente.
#
# Ação:
#
# Constrói a curva de calibração em dez grupos
# com quantidades semelhantes de registros.

from sklearn.calibration import (
    calibration_curve
)


fracao_positivos, media_probabilidades = (
    calibration_curve(
        y_teste_risco,
        probabilidade_teste_risco,
        n_bins=10,
        strategy="quantile"
    )
)


fig, ax = plt.subplots(figsize=(7, 6))

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray",
    label="Calibração perfeita"
)

ax.plot(
    media_probabilidades,
    fracao_positivos,
    marker="o",
    label=nome_modelo_otimizado_selecionado
)

ax.set_title(
    "Curva de calibração — probabilidade da classe 0"
)
ax.set_xlabel("Probabilidade média prevista")
ax.set_ylabel("Frequência observada da classe 0")
ax.grid(alpha=0.25)
ax.legend()

plt.tight_layout()
plt.show()

### 11.5 Intervalos de confiança por bootstrap agrupado

Como alunos do mesmo município compartilham features contextuais, eles não devem ser tratados como observações totalmente independentes na estimação da incerteza.

O bootstrap será realizado no nível municipal: municípios serão sorteados com reposição e todos os alunos associados receberão o mesmo multiplicador. Essa implementação utiliza pesos para evitar a materialização repetida de centenas de milhares de linhas.

Serão executadas 200 reamostragens reproduzíveis. O intervalo de 95% será definido pelos percentis 2,5 e 97,5.

In [0]:
# Objetivo:
#
# Quantificar a incerteza das métricas finais
# respeitando a dependência municipal.
#
# Justificativa:
#
# O bootstrap por aluno subestimaria a incerteza
# ao ignorar a repetição de contexto territorial.
#
# Ação:
#
# Reamostra municípios com reposição e calcula
# intervalos percentis de 95%.

N_BOOTSTRAP = 200
RANDOM_STATE_BOOTSTRAP = 42


grupos_teste = (
    auxiliares_teste["co_municipio"]
    .astype("string")
    .fillna("SEM_MUNICIPIO")
    .reset_index(drop=True)
)


codigos_grupos, municipios_unicos = (
    pd.factorize(
        grupos_teste,
        sort=False
    )
)

quantidade_municipios_teste = len(
    municipios_unicos
)

rng_bootstrap = np.random.default_rng(
    RANDOM_STATE_BOOTSTRAP
)

registros_bootstrap = []


for repeticao in range(N_BOOTSTRAP):
    municipios_sorteados = rng_bootstrap.integers(
        0,
        quantidade_municipios_teste,
        size=quantidade_municipios_teste
    )

    multiplicadores = np.bincount(
        municipios_sorteados,
        minlength=quantidade_municipios_teste
    )

    pesos_linhas = multiplicadores[
        codigos_grupos
    ].astype("float64")

    mascara_presente = pesos_linhas > 0

    y_boot = y_teste.iloc[mascara_presente]
    pred_boot = y_pred_teste_final[
        mascara_presente
    ]
    proba_boot = probabilidade_teste_risco[
        mascara_presente
    ]
    pesos_boot = pesos_linhas[mascara_presente]

    y_boot_risco = y_boot.eq(0).astype("int8")

    if y_boot_risco.nunique() < 2:
        continue

    registros_bootstrap.append({
        "repeticao": repeticao,
        "recall_nao_alfabetizado": recall_score(
            y_boot,
            pred_boot,
            pos_label=0,
            sample_weight=pesos_boot
        ),
        "precision_nao_alfabetizado": precision_score(
            y_boot,
            pred_boot,
            pos_label=0,
            zero_division=0,
            sample_weight=pesos_boot
        ),
        "f1_nao_alfabetizado": f1_score(
            y_boot,
            pred_boot,
            pos_label=0,
            zero_division=0,
            sample_weight=pesos_boot
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_boot,
            pred_boot,
            sample_weight=pesos_boot
        ),
        "pr_auc_risco": average_precision_score(
            y_boot_risco,
            proba_boot,
            sample_weight=pesos_boot
        ),
        "roc_auc_risco": roc_auc_score(
            y_boot_risco,
            proba_boot,
            sample_weight=pesos_boot
        )
    })

    if (repeticao + 1) % 50 == 0:
        print(
            "Bootstrap concluído: "
            f"{repeticao + 1}/{N_BOOTSTRAP}"
        )


resultados_bootstrap = pd.DataFrame(
    registros_bootstrap
)


metricas_bootstrap = [
    coluna
    for coluna in resultados_bootstrap.columns
    if coluna != "repeticao"
]


intervalos_confianca = pd.DataFrame({
    "estimativa_teste": [
        metricas_teste_final[metrica]
        for metrica in metricas_bootstrap
    ],
    "media_bootstrap": [
        resultados_bootstrap[metrica].mean()
        for metrica in metricas_bootstrap
    ],
    "ic_95_inferior": [
        resultados_bootstrap[metrica].quantile(
            0.025
        )
        for metrica in metricas_bootstrap
    ],
    "ic_95_superior": [
        resultados_bootstrap[metrica].quantile(
            0.975
        )
        for metrica in metricas_bootstrap
    ]
}, index=metricas_bootstrap)


intervalos_confianca

### 11.6 Comparação estatística pós-hoc com o melhor modelo alternativo otimizado

A comparação será realizada contra a melhor pipeline **otimizada** de uma família alternativa. Se o HistGradientBoosting for o modelo selecionado, a Random Forest balanceada otimizada será utilizada; caso a Random Forest seja selecionada, a Regressão Logística balanceada otimizada será o comparador.

Essa correção assegura equivalência experimental: modelo final e comparador passaram pelo mesmo processo de otimização e reajuste no treino completo. A análise é confirmatória e pós-hoc, não altera a seleção feita na validação e não autoriza troca do modelo com base no teste.

Serão utilizados o teste exato de McNemar para discordâncias de classificação e um bootstrap municipal pareado para a diferença de PR-AUC.


In [0]:
# Objetivo:
#
# Comparar estatisticamente o modelo congelado
# com a melhor alternativa otimizada.
#
# Justificativa:
#
# Uma comparação justa exige que ambos os modelos
# tenham passado pelo mesmo protocolo de tuning e
# reajuste no treino completo.
#
# A análise é pós-hoc, não altera a seleção realizada
# na validação e não permite reajuste pelo teste.
#
# Ação:
#
# Executa McNemar exato e bootstrap municipal pareado
# para a diferença de PR-AUC.

from scipy.stats import binomtest


if nome_modelo_otimizado_selecionado != (
    "Random_Forest_balanceada"
):
    nome_modelo_comparador = (
        "Random_Forest_balanceada"
    )
else:
    nome_modelo_comparador = (
        "Logistica_balanceada"
    )


if nome_modelo_comparador not in pipelines_otimizadas:
    raise RuntimeError(
        "Pipeline otimizada do comparador não encontrada: "
        f"{nome_modelo_comparador}."
    )


pipeline_comparador = pipelines_otimizadas[
    nome_modelo_comparador
]

pred_comparador = pipeline_comparador.predict(
    X_teste
)

proba_comparador = probabilidade_classe(
    pipeline_comparador,
    X_teste,
    classe_interesse=0
)


acerto_final = (
    y_pred_teste_final == np.asarray(y_teste)
)

acerto_comparador = (
    pred_comparador == np.asarray(y_teste)
)


somente_final_acerta = int(np.sum(
    acerto_final & ~acerto_comparador
))

somente_comparador_acerta = int(np.sum(
    ~acerto_final & acerto_comparador
))


discordancias = (
    somente_final_acerta
    + somente_comparador_acerta
)


if discordancias > 0:
    resultado_mcnemar = binomtest(
        k=min(
            somente_final_acerta,
            somente_comparador_acerta
        ),
        n=discordancias,
        p=0.5,
        alternative="two-sided"
    )
    p_valor_mcnemar = resultado_mcnemar.pvalue
else:
    p_valor_mcnemar = 1.0


diferencas_pr_auc = []
rng_comparacao = np.random.default_rng(42)


for repeticao in range(N_BOOTSTRAP):
    municipios_sorteados = rng_comparacao.integers(
        0,
        quantidade_municipios_teste,
        size=quantidade_municipios_teste
    )

    multiplicadores = np.bincount(
        municipios_sorteados,
        minlength=quantidade_municipios_teste
    )

    pesos_linhas = multiplicadores[
        codigos_grupos
    ].astype("float64")

    mascara = pesos_linhas > 0
    pesos = pesos_linhas[mascara]
    y_risco = y_teste_risco.iloc[mascara]

    if y_risco.nunique() < 2:
        continue

    pr_auc_final = average_precision_score(
        y_risco,
        probabilidade_teste_risco[mascara],
        sample_weight=pesos
    )

    pr_auc_comparador = average_precision_score(
        y_risco,
        proba_comparador[mascara],
        sample_weight=pesos
    )

    diferencas_pr_auc.append(
        pr_auc_final - pr_auc_comparador
    )


diferencas_pr_auc = np.asarray(
    diferencas_pr_auc
)

if diferencas_pr_auc.size == 0:
    raise RuntimeError(
        "O bootstrap pareado não produziu reamostragens válidas."
    )


ic_diferenca_inferior = float(
    np.quantile(diferencas_pr_auc, 0.025)
)

ic_diferenca_superior = float(
    np.quantile(diferencas_pr_auc, 0.975)
)

pr_auc_superioridade_confirmada = bool(
    ic_diferenca_inferior > 0
)


comparacao_estatistica = pd.DataFrame([{
    "modelo_final": (
        nome_modelo_otimizado_selecionado
    ),
    "modelo_comparador": nome_modelo_comparador,
    "configuracao_comparador": "otimizada",
    "somente_modelo_final_acerta": (
        somente_final_acerta
    ),
    "somente_comparador_acerta": (
        somente_comparador_acerta
    ),
    "p_valor_mcnemar": float(p_valor_mcnemar),
    "diferenca_pr_auc_media": float(
        diferencas_pr_auc.mean()
    ),
    "diferenca_pr_auc_ic_95_inferior": (
        ic_diferenca_inferior
    ),
    "diferenca_pr_auc_ic_95_superior": (
        ic_diferenca_superior
    ),
    "superioridade_pr_auc_confirmada": (
        pr_auc_superioridade_confirmada
    ),
    "selecao_alterada_pelo_teste": False,
    "natureza_analise": "pos_hoc_confirmatoria"
}])


display(comparacao_estatistica)


### 11.7 Estabilidade por UF, região e dependência administrativa

O desempenho agregado pode ocultar degradações importantes em determinados grupos. Serão calculadas métricas por UF, região e dependência administrativa, respeitando um mínimo de 100 registros e verificando a presença das duas classes antes das métricas probabilísticas.

Essas análises têm finalidade diagnóstica e de governança; não serão usadas para reajustar o modelo após o teste.

In [0]:
# Objetivo:
#
# Avaliar a estabilidade do modelo final em
# diferentes recortes educacionais e territoriais.
#
# Justificativa:
#
# Uma média global pode esconder baixo desempenho
# em regiões ou grupos específicos.
#
# Ação:
#
# Monta a base de avaliação e calcula métricas
# por UF, região e dependência administrativa.

base_avaliacao_teste = pd.DataFrame({
    "y_real": y_teste.reset_index(drop=True),
    "y_pred": y_pred_teste_final,
    "probabilidade_risco": (
        probabilidade_teste_risco
    ),
    "co_uf": (
        X_teste["co_uf"]
        .reset_index(drop=True)
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
    ),
    "tp_dependencia": (
        X_teste["tp_dependencia"]
        .reset_index(drop=True)
        .astype("string")
    ),
    "co_municipio": grupos_teste
})


mapa_regiao = {
    "11": "Norte", "RO": "Norte",
    "12": "Norte", "AC": "Norte",
    "13": "Norte", "AM": "Norte",
    "14": "Norte", "RR": "Norte",
    "15": "Norte", "PA": "Norte",
    "16": "Norte", "AP": "Norte",
    "17": "Norte", "TO": "Norte",
    "21": "Nordeste", "MA": "Nordeste",
    "22": "Nordeste", "PI": "Nordeste",
    "23": "Nordeste", "CE": "Nordeste",
    "24": "Nordeste", "RN": "Nordeste",
    "25": "Nordeste", "PB": "Nordeste",
    "26": "Nordeste", "PE": "Nordeste",
    "27": "Nordeste", "AL": "Nordeste",
    "28": "Nordeste", "SE": "Nordeste",
    "29": "Nordeste", "BA": "Nordeste",
    "31": "Sudeste", "MG": "Sudeste",
    "32": "Sudeste", "ES": "Sudeste",
    "33": "Sudeste", "RJ": "Sudeste",
    "35": "Sudeste", "SP": "Sudeste",
    "41": "Sul", "PR": "Sul",
    "42": "Sul", "SC": "Sul",
    "43": "Sul", "RS": "Sul",
    "50": "Centro-Oeste", "MS": "Centro-Oeste",
    "51": "Centro-Oeste", "MT": "Centro-Oeste",
    "52": "Centro-Oeste", "GO": "Centro-Oeste",
    "53": "Centro-Oeste", "DF": "Centro-Oeste"
}


base_avaliacao_teste["regiao"] = (
    base_avaliacao_teste["co_uf"]
    .str.upper()
    .map(mapa_regiao)
    .fillna("Não identificada")
)


def metricas_por_grupo(
    dados,
    coluna_grupo,
    minimo_registros=100
):
    linhas = []

    for grupo, recorte in dados.groupby(
        coluna_grupo,
        dropna=False
    ):
        if len(recorte) < minimo_registros:
            continue

        y_real = recorte["y_real"]
        y_pred = recorte["y_pred"]
        y_risco = y_real.eq(0).astype("int8")

        linha = {
            coluna_grupo: grupo,
            "registros": len(recorte),
            "percentual_classe_0": y_risco.mean(),
            "recall_nao_alfabetizado": recall_score(
                y_real,
                y_pred,
                pos_label=0,
                zero_division=0
            ),
            "precision_nao_alfabetizado": precision_score(
                y_real,
                y_pred,
                pos_label=0,
                zero_division=0
            ),
            "f1_nao_alfabetizado": f1_score(
                y_real,
                y_pred,
                pos_label=0,
                zero_division=0
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_real,
                y_pred
            )
        }

        if y_risco.nunique() == 2:
            linha["pr_auc_risco"] = (
                average_precision_score(
                    y_risco,
                    recorte["probabilidade_risco"]
                )
            )
        else:
            linha["pr_auc_risco"] = np.nan

        linhas.append(linha)

    return pd.DataFrame(linhas)


metricas_por_uf = metricas_por_grupo(
    base_avaliacao_teste,
    "co_uf"
)

metricas_por_regiao = metricas_por_grupo(
    base_avaliacao_teste,
    "regiao"
)

metricas_por_dependencia = metricas_por_grupo(
    base_avaliacao_teste,
    "tp_dependencia"
)


print("Métricas por região:")
display(
    metricas_por_regiao.sort_values(
        "recall_nao_alfabetizado"
    )
)

print("Métricas por dependência administrativa:")
display(
    metricas_por_dependencia.sort_values(
        "recall_nao_alfabetizado"
    )
)

print("UFs com menor recall da classe 0:")
display(
    metricas_por_uf.sort_values(
        "recall_nao_alfabetizado"
    ).head(10)
)

### 11.8 Diagnóstico de generalização e estabilidade territorial

Esta etapa registra explicitamente duas conclusões que precisam acompanhar o modelo:

1. o recall mínimo de 70% foi utilizado para congelar o limiar na validação, mas deve ser novamente observado no teste sem qualquer reajuste;
2. o desempenho agregado pode esconder diferenças relevantes entre regiões e UFs.

O diagnóstico calcula a diferença entre validação e teste, identifica a região de menor recall e lista UFs com recall igual a zero. Os resultados são evidências de governança e limitações do ciclo atual. Eles não serão utilizados para alterar o modelo, o limiar ou as features.


In [0]:
# Objetivo:
#
# Documentar a generalização do recall e a
# estabilidade territorial do modelo congelado.
#
# Justificativa:
#
# A restrição operacional foi definida na validação,
# mas sua manutenção deve ser verificada no teste.
# Diferenças entre territórios precisam ser registradas
# para evitar conclusões baseadas somente na média.
#
# Ação:
#
# Calcula o gap validação-teste, resume extremos
# regionais e sinaliza UFs com recall igual a zero.

recall_validacao_congelado = float(
    decisao_modelagem["recall_nao_alfabetizado"]
)

recall_teste_observado = float(
    metricas_teste_final["recall_nao_alfabetizado"]
)

gap_recall_teste_validacao = (
    recall_teste_observado
    - recall_validacao_congelado
)

regiao_menor_recall = (
    metricas_por_regiao
    .sort_values("recall_nao_alfabetizado")
    .iloc[0]
)

regiao_maior_recall = (
    metricas_por_regiao
    .sort_values("recall_nao_alfabetizado")
    .iloc[-1]
)

ufs_recall_zero = (
    metricas_por_uf.loc[
        metricas_por_uf[
            "recall_nao_alfabetizado"
        ].le(1e-12),
        "co_uf"
    ]
    .astype(str)
    .sort_values()
    .tolist()
)

amplitude_recall_regional = float(
    regiao_maior_recall["recall_nao_alfabetizado"]
    - regiao_menor_recall["recall_nao_alfabetizado"]
)


diagnostico_generalizacao = pd.DataFrame([{
    "modelo": nome_modelo_otimizado_selecionado,
    "limiar_congelado": float(LIMIAR_FINAL_VALIDACAO),
    "recall_validacao": recall_validacao_congelado,
    "recall_teste": recall_teste_observado,
    "gap_recall_teste_validacao": float(
        gap_recall_teste_validacao
    ),
    "recall_minimo_desenvolvimento": float(
        RECALL_MINIMO
    ),
    "recall_minimo_mantido_no_teste": bool(
        recall_teste_observado >= RECALL_MINIMO
    ),
    "regiao_menor_recall": str(
        regiao_menor_recall["regiao"]
    ),
    "menor_recall_regional": float(
        regiao_menor_recall[
            "recall_nao_alfabetizado"
        ]
    ),
    "regiao_maior_recall": str(
        regiao_maior_recall["regiao"]
    ),
    "maior_recall_regional": float(
        regiao_maior_recall[
            "recall_nao_alfabetizado"
        ]
    ),
    "amplitude_recall_regional": (
        amplitude_recall_regional
    ),
    "quantidade_ufs_recall_zero": int(
        len(ufs_recall_zero)
    ),
    "ufs_recall_zero": ", ".join(ufs_recall_zero),
    "modelo_ou_limiar_reajustado_pelo_teste": False
}])


display(diagnostico_generalizacao)

print("CONCLUSÃO DE GENERALIZAÇÃO")
print("=" * 70)
print(
    "Recall da classe 0 na validação: "
    f"{recall_validacao_congelado:.2%}"
)
print(
    "Recall da classe 0 no teste: "
    f"{recall_teste_observado:.2%}"
)
print(
    "Diferença teste - validação: "
    f"{gap_recall_teste_validacao:.2%}"
)

if recall_teste_observado < RECALL_MINIMO:
    print(
        "ATENÇÃO: o recall mínimo de desenvolvimento "
        "não se manteve no teste. O limiar não será "
        "reajustado com estes dados."
    )

if amplitude_recall_regional >= 0.20:
    print(
        "ATENÇÃO: existe amplitude regional relevante "
        f"de {amplitude_recall_regional:.2%}."
    )

print(
    "UFs com recall igual a zero: "
    + (", ".join(ufs_recall_zero) or "nenhuma")
)
print(
    "Decisão: preservar modelo e limiar congelados; "
    "registrar as limitações e abrir novo ciclo apenas "
    "se a restrição operacional precisar ser revista."
)


### 11.9 Conclusão da avaliação estatística

A conclusão desta etapa deve registrar as métricas finais, seus intervalos de confiança, a calibração, a comparação estatística e as diferenças entre subgrupos. O desempenho deve ser interpretado como evidência preditiva fora da amostra, e não como relação causal.

O modelo permanece congelado. Qualquer limitação encontrada no teste será documentada no Model Card e orientará um ciclo futuro, sem retroalimentar o experimento atual.


## 12. Interpretabilidade e aplicação estratégica

A interpretabilidade buscará identificar quais features contribuem para as previsões e onde o modelo comete erros. Importância preditiva não representa causalidade: uma variável importante ajuda o modelo a discriminar os casos, mas não prova que sua alteração produzirá melhora na alfabetização.

### 12.1 Permutation Importance

A importância por permutação será calculada sobre uma amostra reprodutível de até 50 mil registros do teste. Cada feature original será embaralhada e a queda na PR-AUC da classe `0` será medida.

Essa abordagem avalia a pipeline completa e mantém os resultados no nível das 16 features originais, facilitando a interpretação.

In [0]:
# Objetivo:
#
# Medir a importância global das features por
# perda de PR-AUC após permutação.
#
# Justificativa:
#
# O método avalia a pipeline completa e não
# depende de um atributo nativo do estimador.
#
# Ação:
#
# Seleciona uma amostra reprodutível e calcula
# cinco permutações por feature.

from sklearn.inspection import (
    permutation_importance
)


TAMANHO_AMOSTRA_INTERPRETABILIDADE = min(
    50_000,
    len(X_teste)
)


indices_interpretabilidade = (
    X_teste.sample(
        n=TAMANHO_AMOSTRA_INTERPRETABILIDADE,
        random_state=42
    ).index
)

X_interpretabilidade = X_teste.loc[
    indices_interpretabilidade
]

y_interpretabilidade = y_teste.loc[
    indices_interpretabilidade
]


resultado_permutacao = permutation_importance(
    estimator=pipeline_modelo_selecionado,
    X=X_interpretabilidade,
    y=y_interpretabilidade,
    scoring=scorer_pr_auc_classe_zero,
    n_repeats=5,
    random_state=42,
    n_jobs=1
)


importancia_permutacao = (
    pd.DataFrame({
        "feature": X_teste.columns,
        "importancia_media": (
            resultado_permutacao.importances_mean
        ),
        "desvio_importancia": (
            resultado_permutacao.importances_std
        )
    })
    .sort_values(
        by="importancia_media",
        ascending=False
    )
    .reset_index(drop=True)
)


display(importancia_permutacao)

In [0]:
# Objetivo:
#
# Visualizar as features mais importantes.
#
# Justificativa:
#
# O gráfico facilita a comunicação dos fatores
# com maior contribuição preditiva global.
#
# Ação:
#
# Exibe as quinze primeiras importâncias com
# suas variações entre permutações.

top_importancias = (
    importancia_permutacao.head(15)
    .sort_values("importancia_media")
)


fig, ax = plt.subplots(figsize=(10, 7))

ax.barh(
    top_importancias["feature"],
    top_importancias["importancia_media"],
    xerr=top_importancias["desvio_importancia"],
    color="#2E74B5",
    alpha=0.85
)

ax.set_title(
    "Permutation Importance — queda na PR-AUC"
)
ax.set_xlabel("Importância média")
ax.grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()

### 12.2 SHAP Values — análise complementar

O SHAP será executado apenas se a biblioteca estiver disponível no runtime. Para o HistGradientBoosting, os valores do `TreeExplainer` representam a saída da classe `1`; por isso, o sinal será invertido para interpretar o aumento de risco da classe `0`.

Será utilizada uma amostra de mil registros. Se o pacote não estiver instalado, a etapa será registrada como pendente sem interromper o restante do notebook. Não se recomenda instalar a biblioteca no meio da execução, pois uma reinicialização do Python apagaria os modelos da memória.

In [0]:
# Objetivo:
#
# Produzir uma interpretação complementar por
# SHAP quando a biblioteca estiver disponível.
#
# Justificativa:
#
# O SHAP permite analisar magnitude e direção
# das contribuições das features transformadas.
#
# Ação:
#
# Verifica a biblioteca, transforma uma amostra
# e produz o summary plot para o modelo esperado.

import importlib.util


SHAP_DISPONIVEL = (
    importlib.util.find_spec("shap") is not None
)


if not SHAP_DISPONIVEL:
    print(
        "Biblioteca shap não disponível. "
        "A Permutation Importance permanece como "
        "método principal de interpretabilidade."
    )
else:
    import shap

    modelo_final = (
        pipeline_modelo_selecionado
        .named_steps["modelo"]
    )

    nome_estimador_final = (
        type(modelo_final).__name__
    )

    if nome_estimador_final != (
        "HistGradientBoostingClassifier"
    ):
        print(
            "O modelo selecionado não é o "
            "HistGradientBoostingClassifier. "
            "Utilize a Permutation Importance como "
            "interpretação principal deste ciclo."
        )
    else:
        X_shap = X_teste.sample(
            n=min(1_000, len(X_teste)),
            random_state=42
        )

        preprocessador_final = (
            pipeline_modelo_selecionado
            .named_steps["pre_processamento"]
        )

        X_shap_transformado = (
            preprocessador_final.transform(X_shap)
        )

        column_transformer_final = (
            preprocessador_final
            .named_steps["pre_processamento"]
        )

        nomes_features_transformadas = (
            column_transformer_final
            .get_feature_names_out()
        )

        explicador_shap = shap.TreeExplainer(
            modelo_final
        )

        valores_shap_classe_1 = (
            explicador_shap.shap_values(
                X_shap_transformado
            )
        )

        valores_shap_risco = -np.asarray(
            valores_shap_classe_1
        )

        shap.summary_plot(
            valores_shap_risco,
            X_shap_transformado,
            feature_names=(
                nomes_features_transformadas
            ),
            max_display=15,
            show=True
        )

### 12.3 Análise de erros e rastreabilidade

As previsões serão reunidas aos identificadores auxiliares, sem introduzi-los como features. Cada registro será classificado como verdadeiro positivo, falso negativo, falso positivo ou verdadeiro negativo sob a perspectiva da classe `0`.

O artefato permitirá investigar erros por aluno, escola e município e apoiar análises posteriores, respeitando regras de privacidade e acesso.

In [0]:
# Objetivo:
#
# Associar previsões e erros aos identificadores
# preservados para rastreabilidade.
#
# Justificativa:
#
# A análise de erros precisa localizar padrões
# sem utilizar os identificadores no treinamento.
#
# Ação:
#
# Constrói o artefato rastreável e resume os
# falsos negativos por município.

resultado_rastreavel_teste = (
    auxiliares_teste.reset_index(drop=True)
    .copy()
)

resultado_rastreavel_teste[
    "y_real"
] = y_teste.reset_index(drop=True)

resultado_rastreavel_teste[
    "probabilidade_nao_alfabetizado"
] = probabilidade_teste_risco

resultado_rastreavel_teste[
    "y_pred"
] = y_pred_teste_final

resultado_rastreavel_teste[
    "limiar_aplicado"
] = LIMIAR_FINAL_VALIDACAO


condicoes_resultado = [
    (
        resultado_rastreavel_teste["y_real"].eq(0)
        & resultado_rastreavel_teste["y_pred"].eq(0)
    ),
    (
        resultado_rastreavel_teste["y_real"].eq(0)
        & resultado_rastreavel_teste["y_pred"].eq(1)
    ),
    (
        resultado_rastreavel_teste["y_real"].eq(1)
        & resultado_rastreavel_teste["y_pred"].eq(0)
    ),
    (
        resultado_rastreavel_teste["y_real"].eq(1)
        & resultado_rastreavel_teste["y_pred"].eq(1)
    )
]

rotulos_resultado = [
    "VP_nao_alfabetizado",
    "FN_nao_alfabetizado",
    "FP_nao_alfabetizado",
    "VN_nao_alfabetizado"
]

resultado_rastreavel_teste[
    "tipo_resultado"
] = np.select(
    condicoes_resultado,
    rotulos_resultado,
    default="não_classificado"
)


falsos_negativos_municipio = (
    resultado_rastreavel_teste.loc[
        resultado_rastreavel_teste[
            "tipo_resultado"
        ] == "FN_nao_alfabetizado"
    ]
    .groupby("co_municipio", dropna=False)
    .size()
    .rename("falsos_negativos")
    .sort_values(ascending=False)
    .reset_index()
)


display(
    falsos_negativos_municipio.head(20)
)

### 12.4 Aplicação estratégica e limites de interpretação

Os resultados podem apoiar a identificação de territórios e grupos que concentram maior risco previsto, orientar a priorização de análises e fundamentar discussões sobre políticas educacionais.

O modelo não deve ser utilizado para rotular definitivamente alunos, aplicar sanções, excluir beneficiários ou substituir avaliação pedagógica. As importâncias representam associações preditivas condicionais aos dados disponíveis e não sustentam afirmações causais.

## 13. Reprodutibilidade e boas práticas produtivas

A pipeline completa será persistida com o pré-processamento, o modelo e seus parâmetros. Também serão registrados ambiente, schema, limiar, métricas, intervalos de confiança, decisões e limitações.

O módulo `src.preprocessing` deverá permanecer disponível no mesmo caminho quando o artefato for recarregado, pois a pipeline contém o transformador customizado `IndicadoresAusencia`.

### 13.1 Registro técnico do experimento

Esta etapa captura versões das principais bibliotecas, configuração do modelo, features esperadas e data de geração em UTC. Os dados não serão incorporados ao artefato do modelo.

In [0]:
# Objetivo:
#
# Registrar ambiente, configuração e contrato
# técnico do experimento final.
#
# Justificativa:
#
# A reprodução exige conhecer versões, parâmetros,
# features, caminhos e limiar utilizados.
#
# Ação:
#
# Cria metadados serializáveis do modelo final.

import hashlib
import platform
from datetime import datetime, timezone

import joblib
import sklearn


ARTEFATOS_PATH = f"{FASE3_ROOT}/artifacts"
RELATORIOS_PATH = f"{FASE3_ROOT}/reports/modelagem"

dbutils.fs.mkdirs(ARTEFATOS_PATH)
dbutils.fs.mkdirs(RELATORIOS_PATH)


parametros_modelo_final = (
    pipeline_modelo_selecionado
    .named_steps["modelo"]
    .get_params(deep=False)
)


def tornar_json_serializavel(valor):
    if isinstance(valor, np.generic):
        return valor.item()

    if isinstance(valor, np.ndarray):
        return valor.tolist()

    if isinstance(valor, (str, int, float, bool)):
        return valor

    if valor is None:
        return None

    return str(valor)


metadados_modelo = {
    "projeto": "Tech Challenge - Fase 3",
    "modelo": nome_modelo_otimizado_selecionado,
    "target": "in_alfabetizado",
    "classe_interesse": 0,
    "limiar": float(LIMIAR_FINAL_VALIDACAO),
    "features": X_treino.columns.tolist(),
    "quantidade_features_entrada": int(
        X_treino.shape[1]
    ),
    "registros_treino": int(len(X_treino)),
    "registros_validacao": int(len(X_validacao)),
    "registros_teste": int(len(X_teste)),
    "parametros_estimador": {
        chave: tornar_json_serializavel(valor)
        for chave, valor in (
            parametros_modelo_final.items()
        )
    },
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "gerado_em_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "teste_utilizado": True,
    "resultado_teste_nao_usado_para_reajuste": True
}


assinatura_configuracao = hashlib.sha256(
    json.dumps(
        metadados_modelo,
        ensure_ascii=False,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()

metadados_modelo[
    "assinatura_configuracao_sha256"
] = assinatura_configuracao


pd.Series({
    "modelo": metadados_modelo["modelo"],
    "limiar": metadados_modelo["limiar"],
    "scikit_learn": (
        metadados_modelo["scikit_learn"]
    ),
    "assinatura": assinatura_configuracao
})

### 13.2 Persistência da pipeline e dos resultados finais

Serão persistidos a pipeline completa, os metadados, o schema, as métricas, os intervalos, as análises de estabilidade, as importâncias e o artefato rastreável de previsões.

A gravação ocorrerá exclusivamente na estrutura da Fase 3.

In [0]:
# Objetivo:
#
# Persistir o modelo completo e os artefatos
# finais da avaliação.
#
# Justificativa:
#
# Salvar apenas o estimador perderia imputação,
# encoding e engenharia de atributos.
#
# Ação:
#
# Grava pipeline, metadados, métricas, schema,
# relatórios e previsões rastreáveis.

PIPELINE_FILE = (
    f"{ARTEFATOS_PATH}/model_pipeline.joblib"
)

METADATA_FILE = (
    f"{ARTEFATOS_PATH}/model_metadata.json"
)

SCHEMA_FILE = (
    f"{ARTEFATOS_PATH}/feature_schema.json"
)


joblib.dump(
    pipeline_modelo_selecionado,
    PIPELINE_FILE
)


with open(
    METADATA_FILE,
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        metadados_modelo,
        arquivo,
        ensure_ascii=False,
        indent=2
    )


schema_features = {
    coluna: str(tipo)
    for coluna, tipo in X_treino.dtypes.items()
}


with open(
    SCHEMA_FILE,
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        schema_features,
        arquivo,
        ensure_ascii=False,
        indent=2
    )


metricas_teste_final.to_frame(
    "valor"
).to_csv(
    f"{RELATORIOS_PATH}/metricas_teste_final.csv",
    sep=";",
    encoding="utf-8-sig"
)

diagnostico_generalizacao.to_csv(
    f"{RELATORIOS_PATH}/diagnostico_generalizacao.csv",
    sep=";",
    encoding="utf-8-sig",
    index=False
)

comparacao_estatistica.to_csv(
    f"{RELATORIOS_PATH}/comparacao_estatistica_otimizada.csv",
    sep=";",
    encoding="utf-8-sig",
    index=False
)

intervalos_confianca.to_csv(
    f"{RELATORIOS_PATH}/intervalos_confianca.csv",
    sep=";",
    encoding="utf-8-sig"
)

resultados_bootstrap.to_parquet(
    f"{RELATORIOS_PATH}/resultados_bootstrap.parquet",
    index=False
)

metricas_por_uf.to_csv(
    f"{RELATORIOS_PATH}/metricas_por_uf.csv",
    sep=";",
    encoding="utf-8-sig",
    index=False
)

metricas_por_regiao.to_csv(
    f"{RELATORIOS_PATH}/metricas_por_regiao.csv",
    sep=";",
    encoding="utf-8-sig",
    index=False
)

metricas_por_dependencia.to_csv(
    f"{RELATORIOS_PATH}/metricas_por_dependencia.csv",
    sep=";",
    encoding="utf-8-sig",
    index=False
)

importancia_permutacao.to_csv(
    f"{RELATORIOS_PATH}/importancia_permutacao.csv",
    sep=";",
    encoding="utf-8-sig",
    index=False
)

resultado_rastreavel_teste.to_parquet(
    f"{RELATORIOS_PATH}/previsoes_teste_rastreaveis.parquet",
    index=False
)


pd.Series({
    "pipeline": PIPELINE_FILE,
    "metadados": METADATA_FILE,
    "schema": SCHEMA_FILE,
    "relatorios": RELATORIOS_PATH
})


### 13.3 Contrato de inferência

Antes de produzir previsões, novos dados deverão conter exatamente as features esperadas. Colunas auxiliares ou desconhecidas não serão aceitas silenciosamente, e a ausência de qualquer feature obrigatória interromperá a inferência.

A validação dos tipos será informativa nesta versão, pois algumas fontes podem alternar tipos numéricos equivalentes. Conversões de tipo deverão ser tratadas em uma evolução controlada do contrato.

In [0]:
# Objetivo:
#
# Criar uma validação mínima do schema de entrada.
#
# Justificativa:
#
# A inferência não deve prosseguir com features
# ausentes, extras ou em ordem desconhecida.
#
# Ação:
#
# Valida colunas e retorna a entrada na ordem
# utilizada durante o treinamento.

FEATURES_ESPERADAS = X_treino.columns.tolist()


def validar_entrada_inferencia(dados):
    colunas_recebidas = set(dados.columns)
    colunas_esperadas = set(FEATURES_ESPERADAS)

    ausentes = sorted(
        colunas_esperadas - colunas_recebidas
    )

    extras = sorted(
        colunas_recebidas - colunas_esperadas
    )

    if ausentes or extras:
        raise ValueError(
            "Contrato de entrada inválido. "
            f"Ausentes: {ausentes}. "
            f"Extras: {extras}."
        )

    return dados.loc[:, FEATURES_ESPERADAS].copy()


amostra_contrato = validar_entrada_inferencia(
    X_teste.head(100)
)


pd.Series({
    "colunas_recebidas": amostra_contrato.shape[1],
    "colunas_esperadas": len(FEATURES_ESPERADAS),
    "ordem_validada": (
        amostra_contrato.columns.tolist()
        == FEATURES_ESPERADAS
    )
})

### 13.4 Teste de recarga e inferência ponta a ponta

O artefato persistido será recarregado em memória e aplicado a cem registros. As probabilidades deverão ser idênticas às produzidas pela pipeline original, dentro da tolerância numérica.

In [0]:
# Objetivo:
#
# Validar a recarga da pipeline persistida.
#
# Justificativa:
#
# Um arquivo salvo não é considerado válido até
# produzir inferência ponta a ponta após recarga.
#
# Ação:
#
# Recarrega o joblib, calcula probabilidades e
# compara com a pipeline mantida em memória.

pipeline_recarregada = joblib.load(
    PIPELINE_FILE
)


amostra_inferencia = validar_entrada_inferencia(
    X_teste.head(100)
)

proba_original = probabilidade_classe(
    pipeline_modelo_selecionado,
    amostra_inferencia,
    classe_interesse=0
)

proba_recarregada = probabilidade_classe(
    pipeline_recarregada,
    amostra_inferencia,
    classe_interesse=0
)


recarga_valida = np.allclose(
    proba_original,
    proba_recarregada,
    rtol=1e-10,
    atol=1e-12
)


assert recarga_valida, (
    "A pipeline recarregada não reproduziu "
    "as probabilidades originais."
)


pd.Series({
    "pipeline_recarregada": True,
    "registros_testados": len(amostra_inferencia),
    "probabilidades_equivalentes": recarga_valida
})

### 13.5 Model Card

O Model Card documentará finalidade, população, dados, métrica principal, limiar, desempenho, limitações, usos permitidos e usos proibidos. O conteúdo será gerado dinamicamente a partir dos resultados finais.

In [0]:
# Objetivo:
#
# Criar o Model Card do artefato final.
#
# Justificativa:
#
# A documentação de governança esclarece o uso
# responsável e as limitações da solução.
#
# Ação:
#
# Gera um arquivo Markdown com informações
# técnicas e orientações de uso.

MODEL_CARD_FILE = (
    f"{RELATORIOS_PATH}/MODEL_CARD.md"
)


model_card = f"""# Model Card — Tech Challenge Fase 3

## Identificação

- Modelo: {nome_modelo_otimizado_selecionado}
- Target: in_alfabetizado
- Classe de interesse: 0 — não alfabetizado
- Limiar: {LIMIAR_FINAL_VALIDACAO:.4f}
- Data UTC: {metadados_modelo['gerado_em_utc']}

## Finalidade

Estimar risco de não alfabetização para apoiar análises e priorização de políticas educacionais.

## Dados

- Treino: {len(X_treino):,} registros
- Validação: {len(X_validacao):,} registros
- Teste: {len(X_teste):,} registros
- Separação territorial por município
- {X_treino.shape[1]} features de entrada

## Desempenho no teste

- PR-AUC: {metricas_teste_final['pr_auc_risco']:.6f}
- ROC-AUC: {metricas_teste_final['roc_auc_risco']:.6f}
- Recall da classe 0: {metricas_teste_final['recall_nao_alfabetizado']:.6f}
- Precisão da classe 0: {metricas_teste_final['precision_nao_alfabetizado']:.6f}
- F1 da classe 0: {metricas_teste_final['f1_nao_alfabetizado']:.6f}
- Balanced accuracy: {metricas_teste_final['balanced_accuracy']:.6f}
- Brier Score: {metricas_teste_final['brier_score_risco']:.6f}

## Generalização e estabilidade

- Recall na validação: {recall_validacao_congelado:.6f}
- Recall no teste: {recall_teste_observado:.6f}
- Gap teste - validação: {gap_recall_teste_validacao:.6f}
- Recall mínimo de desenvolvimento mantido no teste: {recall_teste_observado >= RECALL_MINIMO}
- Região de menor recall: {regiao_menor_recall['regiao']} ({regiao_menor_recall['recall_nao_alfabetizado']:.6f})
- Região de maior recall: {regiao_maior_recall['regiao']} ({regiao_maior_recall['recall_nao_alfabetizado']:.6f})
- UFs com recall igual a zero: {', '.join(ufs_recall_zero) or 'nenhuma'}
- O teste não foi utilizado para reajustar modelo, features ou limiar.

## Comparação pós-hoc

- Comparador: {nome_modelo_comparador} otimizado
- Diferença média de PR-AUC: {comparacao_estatistica.loc[0, 'diferenca_pr_auc_media']:.6f}
- IC 95% da diferença: [{comparacao_estatistica.loc[0, 'diferenca_pr_auc_ic_95_inferior']:.6f}, {comparacao_estatistica.loc[0, 'diferenca_pr_auc_ic_95_superior']:.6f}]
- Superioridade em PR-AUC confirmada: {comparacao_estatistica.loc[0, 'superioridade_pr_auc_confirmada']}
- A comparação não altera a seleção realizada na validação.

## Uso permitido

- Apoio à análise de risco educacional.
- Priorização de investigações e ações complementares.
- Análises agregadas com controle de cobertura e incerteza.

## Uso proibido

- Rotular definitivamente um aluno.
- Substituir avaliação pedagógica ou decisão humana qualificada.
- Aplicar sanções, restringir direitos ou excluir beneficiários.
- Interpretar importância preditiva como causalidade.

## Limitações

- Features contextuais municipais se repetem entre alunos.
- O modelo depende da qualidade e temporalidade das fontes públicas.
- Desempenho pode variar por UF, região e dependência administrativa.
- Probabilidades exigem monitoramento de calibração em novos períodos.
- Mudanças de distribuição exigem nova validação antes do uso.

## Monitoramento recomendado

- Schema, ausências e categorias desconhecidas.
- Drift das features e das probabilidades.
- Recall, precisão, PR-AUC e calibração quando o target estiver disponível.
- Desempenho por subgrupos territoriais e educacionais.
"""


with open(
    MODEL_CARD_FILE,
    "w",
    encoding="utf-8"
) as arquivo:
    arquivo.write(model_card)


print(f"Model Card salvo em: {MODEL_CARD_FILE}")


## 14. Estrutura final do projeto

Ao término deste notebook, a estrutura funcional esperada é:

```text
tech_challenge_fase3/
├── src/
│   ├── __init__.py
│   └── preprocessing.py
├── modelagem/
│   ├── treino.parquet
│   ├── validacao.parquet
│   ├── teste.parquet
│   ├── auxiliares_*.parquet
│   └── resultados/
├── artifacts/
│   ├── model_pipeline.joblib
│   ├── model_metadata.json
│   └── feature_schema.json
└── reports/
    └── modelagem/
        ├── metricas_teste_final.csv
        ├── diagnostico_generalizacao.csv
        ├── comparacao_estatistica_otimizada.csv
        ├── intervalos_confianca.csv
        ├── resultados_bootstrap.parquet
        ├── metricas_por_*.csv
        ├── importancia_permutacao.csv
        ├── previsoes_teste_rastreaveis.parquet
        └── MODEL_CARD.md
```

No repositório Git, deverão permanecer notebooks, código-fonte, documentação, requisitos e relatórios leves. Bases, previsões detalhadas e artefatos binários grandes devem permanecer no Volume e ser ignorados pelo Git.


## 15. Critério de encerramento

A modelagem pode ser considerada tecnicamente concluída quando:

1. o modelo e o limiar foram definidos sem utilizar o teste;
2. o teste foi acessado uma única vez;
3. métricas e intervalos de confiança foram registrados;
4. a estabilidade territorial foi avaliada;
5. a interpretabilidade foi produzida sem afirmações causais;
6. a pipeline completa foi persistida e recarregada com sucesso;
7. o schema e o contrato de entrada foram documentados;
8. limitações e usos proibidos constam no Model Card;
9. nenhum resultado do teste foi utilizado para reajustar o experimento atual.

## 17. Checklist final de execução

- [ ] Modelo, hiperparâmetros e limiar congelados antes do teste.
- [ ] Avaliação final executada uma única vez.
- [ ] PR-AUC, ROC-AUC, recall, precisão, F1 e balanced accuracy registrados.
- [ ] Brier Score e curva de calibração analisados.
- [ ] Bootstrap agrupado por município concluído.
- [ ] Comparação estatística entre pipelines otimizadas interpretada.
- [ ] Gap de recall entre validação e teste documentado.
- [ ] Instabilidade territorial registrada no Model Card.
- [ ] Métricas por UF, região e dependência avaliadas.
- [ ] Permutation Importance concluída.
- [ ] SHAP executado ou indisponibilidade documentada.
- [ ] Falsos negativos e falsos positivos rastreáveis.
- [ ] Pipeline completa salva em `artifacts`.
- [ ] Metadados e schema persistidos.
- [ ] Teste de recarga aprovado.
- [ ] Model Card gerado.
- [ ] Arquivos grandes excluídos do versionamento Git.
- [ ] Limitações comunicadas à banca e aos stakeholders.


## 18. Referências técnicas e conclusão executiva

### Referências

- Enunciado do Tech Challenge — Fase 3.
- Guia de Modelagem — Tech Challenge — Fase 3.
- Scikit-learn: Pipeline, model selection, metrics, calibration e inspection.
- SHAP: documentação de interpretabilidade de modelos.
- Documentação interna da construção da base, EDA e decisões de data leakage.

### Conclusão executiva

O fluxo construído transforma a base Gold da Fase 2 em uma solução preditiva reproduzível na Fase 3. A separação territorial, o pré-processamento encapsulado, a otimização agrupada, o limiar orientado ao risco, a avaliação independente e a quantificação de incerteza reduzem resultados excessivamente otimistas e tornam as decisões auditáveis.

O modelo deve ser utilizado como instrumento de apoio à priorização e à investigação, nunca como decisão automática ou evidência causal. Seu valor estratégico está em antecipar padrões de risco, localizar limitações e apoiar gestores educacionais com evidências transparentes, rastreáveis e passíveis de monitoramento.

In [0]:
# Objetivo:
#
# Consolidar o status final dos artefatos.
#
# Justificativa:
#
# Uma validação final facilita a entrega e evita
# ausência silenciosa de arquivos essenciais.
#
# Ação:
#
# Confirma a existência dos principais artefatos
# e apresenta o encerramento técnico.

artefatos_essenciais = {
    "pipeline": PIPELINE_FILE,
    "metadados": METADATA_FILE,
    "schema": SCHEMA_FILE,
    "model_card": MODEL_CARD_FILE,
    "metricas_teste": (
        f"{RELATORIOS_PATH}/metricas_teste_final.csv"
    ),
    "intervalos_confianca": (
        f"{RELATORIOS_PATH}/intervalos_confianca.csv"
    ),
    "previsoes_rastreaveis": (
        f"{RELATORIOS_PATH}/"
        "previsoes_teste_rastreaveis.parquet"
    )
}


validacao_artefatos_finais = pd.DataFrame({
    "artefato": list(artefatos_essenciais.keys()),
    "caminho": list(artefatos_essenciais.values()),
    "existe": [
        os.path.exists(caminho)
        for caminho in artefatos_essenciais.values()
    ]
})


if not validacao_artefatos_finais["existe"].all():
    raise RuntimeError(
        "Existem artefatos finais ausentes. "
        "Revise as etapas de persistência."
    )


display(validacao_artefatos_finais)

print(
    "Modelagem concluída com avaliação final, "
    "rastreabilidade e artefatos validados."
)